# RSNA Knee Abnormality Detection / `RSNA Knee | DINOsaur V3 🦖`

- **コンペ**: [RSNA Knee Abnormality Detection](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection)（残り約2か月・2000チーム規模）
- **原著notebook**: [RSNA Knee | DINOsaur V3 🦖](https://www.kaggle.com/code/romantamrazov/rsna-knee-dinosaur-v3)
- **原著者**: ROMAN TAMRAZOV (romantamrazov) ・ 57 votes（Bronze）・ Apache 2.0 ・ Version 20 / ランタイム 21分57秒（GPU T4 x2）
- **スコア**: Best 0.921 V18（**執筆時点の公開LB最上位帯**）

> ⚠️ これは**学習目的の解説付き写し**です。コード本体は原著のまま変更していませんが、実行結果（outputs）は含みません。
> コードの著作権は原著者に帰属します（Apache 2.0）。
> このnotebookは3つの巨大なコードセル（合計約18万文字・関数/クラス108個）でできています。
> 全部を読む必要はありません。**設計の骨格と、そこに込められた判断**を追ってください。

## 手法の概要 — 「3つの独立したパイプラインを、順位で合流させる」

膝MRIの検査1件（複数シリーズ＝矢状断・冠状断・軸位断 × 脂肪抑制あり/なし）から、
**12種類の異常**（ACL/MCL断裂、半月板、変形性関節症3部位、関節液貯留、滑膜炎、ベーカー嚢腫、挫傷、骨折）を
同時に判定する**マルチラベル分類**です。

このnotebookの構造は、1つの巨大モデルではなく**3つの独立した予測系を順に走らせて最後に混ぜる**というものです。

| セル | パイプライン | 特徴づけ |
|---|---|---|
| 1 | **DINOv2 フロンティア系** | 自然画像で自己教師あり学習した DINOv2-small を部分ファインチューニング。スロット設計・キャッシュ・時間予算管理が中心 |
| 2 | **A5 / Net 系（自作アーキテクチャ）** | timm系バックボーン＋独自のプーリング群（LabelAttentionPool, SlotDepthMixer, GatedDelta など）でスライス方向を集約 |
| 3 | **RadImageNet + メタスタック系（v10→v11→v12）** | 医用画像で事前学習した ResNet-50 の特徴に、段階的にメタモデルを重ねて挑戦者(challenger)を立てる |

**共通する思想は「間違え方の違うモデルを作り、確率ではなく順位で混ぜる」**です。
前作 V2 からの主な進化は、(a) `_v12_*` 系の新しいメタチャレンジャー層、(b) fold一貫性・
ブートストラップによる**採用ゲート**の追加、(c) E11/E13 という別スロット設計のメンバー追加です。

## 評価指標

- **タスク**: 検査単位・ラベル単位の**マルチラベル2値分類**（12ラベルが同時に立ちうる）。
- **指標**: **12ラベルのAUCのマクロ平均**（各ラベルで独立にAUCを計算し、単純平均）。
- **なぜこの指標か**:
  - 各異常の**有病率が極端に違います**（変形性関節症は多く、骨折は稀）。マイクロ平均や accuracy だと
    頻度の高いラベルだけで点数が決まり、稀だが臨床的に重い所見（骨折）を当てても報われません。
    **マクロ平均は12ラベルを対等に扱う**ので、稀なラベルの改善がそのまま点になります。
  - AUCは**順位ベース**なので、施設ごとの撮像プロトコル差で予測値のスケールがずれていても、
    順序さえ保たれていれば減点されません。多施設データに適した選択です。
- **この手法が指標をどう最適化しているか**:
  1. **ラベルごとに順位化してから混ぜる**（`_rad_rank_columns`, `rank(pct=True)`）。
     マクロ平均AUCは**ラベルごとに独立**なので、混ぜる操作もラベルごとに独立に行うのが理にかなっています。
     ラベル横断で正規化してしまうと、有病率の違いが順位を歪めます。
  2. **ラベルごとに重みを変える**（`_v10_target_auc` → `v12_weight` は12次元のベクトル）。
     全ラベル共通の1つの重みではなく、**当てやすいラベルと当てにくいラベルで最適な混ぜ方が違う**ことを前提にしています。
  3. **稀ラベルは除外する安全弁**（`_RAD_EXCLUDE = ("Baker's", "Fracture")`）。
     陽性が少ないラベルではメタモデルの学習自体がノイズになりうるため、そこだけベース予測を使います。
  4. **採用ゲート**（`_v10_bootstrap_positive_fraction`, `_v11_fold_consistency`）。
     「AUCが上がったから採用」ではなく、**ブートストラップで改善が正の割合**と
     **fold間で一貫して改善しているか**を確認してから採用します。
     Playground枠の解説と同じ問題意識——**見かけの改善と本物の改善を区別する**——がここでも実装されています。


### 【解説】セル1: DINOv2 フロンティア — スロット設計・キャッシュ・時間予算

**何をしているか**: DICOMを読み、検査ごとに「どのシリーズをどの役割（スロット）に割り当てるか」を決め、
画像をキャッシュし、DINOv2-small ベースのモデルで推論して1本目の提出候補を作ります。関数38個。

**なぜそうするのか（読みどころ5つ）**:

**① スロット設計（`SLOTS_RECOVERED` / `SLOTS_PUBLIC`）**
膝MRIは1検査に何本もシリーズがあり、**本数も種類も検査ごとにバラバラ**です。
モデルの入力は固定長でなければならないので、「矢状断・脂肪抑制あり」「冠状断・T1」…という
**役割の枠（スロット）を先に決めて、そこに埋める**設計にします。
`recovered` はDICOMヘッダから脂肪抑制の有無まで推定した細かい版、`public` は粗い版で、
環境変数で切り替えられます。**枠が埋まらなければ空のまま**にしており、
「近いシリーズで代用する」ことをしません（代用禁止の原則）。ここが重要で、
別種類のシーケンスを代用すると、モデルが「見たことのない組み合わせ」を見ることになり予測が壊れます。

**② 物理サイズ基準のクロップ（`CROP_MM = 130.0`）**
「画像の中央から何ピクセル」ではなく「**中央から130mm四方**」を切り出します。
MRIはDICOMの `PixelSpacing` に1ピクセルの実寸(mm)が入っているので、装置や視野が違っても
**常に同じ解剖学的範囲**を切り出せます。ピクセル基準で切ると、視野の広い装置では膝が小さく写り、
モデルにとって別物になってしまいます。

**③ 左右（laterality）の幾何学的な復元（`side_from_geometry`, `LAT_MIN_OFFSET_MM`）**
右膝か左膝かはヘッダに書かれていないことがあり、`ImagePositionPatient`（患者座標系での位置）から
推定しています。内側/外側（medial/lateral）を区別する半月板や変形性関節症のラベルは、
**左右を間違えると内外が反転して正解と逆になる**ため、ここを間違えると特定ラベルだけスコアが崩壊します。
`RULES_NATIVE` / `RULES_LEGACY` という2つのルールセットが用意されているのは、
**過去バージョンとの互換性を保ちつつ新しい復元法へ移行する**ためです（設定ドリフト対策）。

**④ キャッシュ予算と時間予算（`CACHE_BUDGET_GB`, `TIME_BUDGET = 8時間`, `ORDER_BUDGET_S`）**
コードコンペには実行時間の上限があり、**時間切れ＝提出ゼロ**です。
そのため「使えるRAMの45%までしかキャッシュしない」「スライス順序の解決に使う時間は5400秒まで」といった
**予算**を明示的に管理しています。精度を上げる工夫と同じくらい、**完走させる工夫**が重要だという実例です。

**⑤ 指紋照合（`fingerprint`, `check_fingerprint`, `FINGERPRINT_TOL`）**
学習済み重みを読み込んだあと、**決まった入力を通して出力が期待値と一致するか**を確認します。
`WeightsError` を投げる設計なので、重みの取り違えや前処理の食い違いがあれば
**間違った予測を出す前に止まります**。長時間ジョブでは、静かに間違えるのが最悪です。

**用語**: *DINOv2* = Meta が公開した自己教師あり学習の視覚基盤モデル。ラベルなしの自然画像だけで
汎用的な特徴抽出を学んでおり、少ないラベルでも医用画像に転移しやすい。
*部分ファインチューニング* = バックボーンの大半を凍結し、最後の数ブロックだけ学習する手法
（`unfreeze_last`）。データが少ないときの過学習を防ぎつつ、ドメイン差に適応させる定石です。


In [ ]:
from __future__ import annotations
import os
import gc
import hashlib
import json
import re
import time
import traceback
import threading
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
ASSET = Path('/kaggle/input/datasets/tonylica/rsna-knee-bend-dinov3-0917-repro-assets')
ROOT = Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')
DINO = Path('/kaggle/input/models/metaresearch/dinov2/pytorch/small/1')
T0 = time.time()
DEVS = [torch.device(f'cuda:{i}') for i in range(torch.cuda.device_count())]
SEED = 2026
TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
CROP_MM = 130.0
CACHE_IMG = 336
GROUP = 3
N_GROUP_MAX = 1
CACHE_FRACTION = 0.45
CACHE_BUDGET_MAX_GB = 24.0
CACHE_BUDGET_GB = 12.0
TEST_SHARE = 0.3
HDR_THREADS = 16
PIX_THREADS = 12
ORDER_THREADS = 32
ORDER_BUDGET_S = 5400
AUG_ROT_DEG = 8.0
AUG_SCALE = 0.08
AUG_SHIFT = 0.05
AUG_INTENSITY = 0.1
LAT_MIN_OFFSET_MM = 20.0
SLICE_BAND = (0.2, 0.8)
RULES_NATIVE = {'order': 'normal', 'lat': 'centre', 'slot_fallback': False, 'decode_fill': 'nearest'}
RULES_LEGACY = {'order': 'dominant_axis', 'lat': 'corner_x', 'slot_fallback': True, 'decode_fill': 'zero'}
RULES = dict(RULES_NATIVE)
LEGACY_LAT_OFFSET_MM = 5.0
EVAL_BATCH = 8
TIME_BUDGET = 8.0 * 3600
SLOTS_RECOVERED = [('SAG_FLUID_FS', 'Sagittal', True, True), ('COR_FLUID_FS', 'Coronal', True, True), ('AX_FLUID_FS', 'Axial', True, True), ('SAG_FLUID_NOFS', 'Sagittal', True, False), ('COR_T1', 'Coronal', False, False), ('SAG_T1', 'Sagittal', False, False)]
SLOTS_PUBLIC = [('SAG_FLUID', 'Sagittal', None, True), ('COR_FLUID', 'Coronal', None, True), ('AX_FLUID', 'Axial', None, True), ('SAG_STRUCT', 'Sagittal', None, False), ('COR_STRUCT', 'Coronal', None, False), ('AX_STRUCT', 'Axial', None, False)]
SLOT_SCHEME = os.environ.get('SLOT_SCHEME', 'recovered')
SLOTS = SLOTS_PUBLIC if SLOT_SCHEME == 'public' else SLOTS_RECOVERED
N_SLOT = len(SLOTS)
POOL_PARTS = {'cls_mean': 2, 'cls_mean_focal': 3}
SLOT_PRIOR_TABLE = {'ACL': (0, 3, 5), 'MCL': (1, 4), 'Medial Meniscus': (0, 1, 3, 4), 'Lateral Meniscus': (0, 1, 3, 4), 'Medial OA': (1, 4, 5), 'Lateral OA': (1, 4, 5), 'PF OA': (0, 2, 5), 'Effusion': (0, 2), 'Synovitis': (0, 2), "Baker's": (0,), 'Contusion': (0, 1, 2), 'Fracture': (0, 1, 2, 4, 5)}
SLOT_PRIOR_STRENGTH = 0.55
FATSAT_OPTS = {'FS', 'FATSAT', 'FAT_SAT', 'FSAT'}
_SEP = re.compile('[_\\-.]')
_FATSAT_RX = re.compile('\\bfs\\b|fatsat|fat sat|\\bstir\\b|\\bspair\\b|\\bspir\\b|\\bwe\\b|water excit|\\btirm\\b|\\bsting\\b|\\bfatsup\\b')
_T1_RX = re.compile('\\bt1\\b|\\bt1w\\b')
_T2_RX = re.compile('\\bt2\\b|\\bt2w\\b')
_PD_RX = re.compile('\\bpd\\b|\\bpdw\\b|proton|\\bdp\\b|dens')

def log(msg):
    print(f'[{time.time() - T0:7.1f}s] {msg}', flush=True)
IMG = CACHE_IMG

def available_gb():
    try:
        with open('/proc/meminfo') as fh:
            info = {k.strip(): v for k, v in (l.split(':', 1) for l in fh if ':' in l)}
        return int(info['MemAvailable'].split()[0]) / 1024 ** 2
    except Exception:
        return CACHE_BUDGET_GB / CACHE_FRACTION

def plan_cache(n_study, n_test=0):
    avail = available_gb()
    budget = min(avail * CACHE_FRACTION, CACHE_BUDGET_MAX_GB)
    n_total = n_study + max(n_test, int(TEST_SHARE * n_study))
    per_slice = n_total * N_SLOT * IMG * IMG
    afford = int(budget * 1024 ** 3 // max(per_slice, 1))
    groups = max(1, min(N_GROUP_MAX, afford // GROUP))
    log(f'memory: {avail:.1f} GB available, {budget:.1f} GB to the cache; sizing for {n_study} train + {n_total - n_study} test studies -> {groups} group(s) of {GROUP} = {groups * GROUP} slices per slot' + (f' (wanted {N_GROUP_MAX})' if groups < N_GROUP_MAX else ''))
    return groups
N_GROUP = plan_cache(len(pd.read_csv(ROOT / 'train.csv')), len(pd.read_csv(ROOT / 'test.csv')))
CACHE_SLICES = GROUP * N_GROUP
HDR_TAGS = ['SeriesDescription', 'SequenceName', 'ScanOptions', 'ScanningSequence', 'RepetitionTime', 'EchoTime', 'Laterality', 'PixelSpacing', 'Rows', 'Columns', 'RescaleSlope', 'RescaleIntercept', 'ImagePositionPatient', 'ImageOrientationPatient', 'Manufacturer', 'ManufacturerModelName', 'MagneticFieldStrength', 'SliceThickness', 'SpacingBetweenSlices', 'FlipAngle', 'EchoTrainLength', 'PercentSampling', 'PercentPhaseFieldOfView']

def _hdr_vec(s, n):
    if not isinstance(s, str):
        return None
    try:
        v = [float(x) for x in s.split('|')]
    except ValueError:
        return None
    return np.array(v) if len(v) >= n else None

def side_from_geometry(h):
    cx = {}
    for r in h.itertuples(index=False):
        ipp = _hdr_vec(getattr(r, 'ImagePositionPatient', None), 3)
        iop = _hdr_vec(getattr(r, 'ImageOrientationPatient', None), 6)
        ps = _hdr_vec(getattr(r, 'PixelSpacing', None), 2)
        rows, cols = (getattr(r, 'Rows', None), getattr(r, 'Columns', None))
        if ipp is None or iop is None or ps is None or (not rows) or (not cols):
            continue
        try:
            c = ipp[:3] + iop[:3] * ps[1] * float(cols) / 2 + iop[3:6] * ps[0] * float(rows) / 2
        except (TypeError, ValueError):
            continue
        cx.setdefault(r.StudyInstanceUID, []).append(float(c[0]))
    out = {}
    for st, xs in cx.items():
        m = float(np.median(xs))
        out[st] = None if abs(m) < LAT_MIN_OFFSET_MM else 'R' if m < 0 else 'L'
    return out

def side_from_corner_x(h):
    out = {}
    for st, g in h.groupby('StudyInstanceUID'):
        xs = []
        for r in g.itertuples(index=False):
            ipp = _hdr_vec(getattr(r, 'ImagePositionPatient', None), 3)
            if ipp is not None and np.isfinite(ipp).all():
                xs.append(float(ipp[0]))
        if not xs:
            out[st] = None
            continue
        x = float(np.median(xs))
        out[st] = None if abs(x) < LEGACY_LAT_OFFSET_MM else 'R' if x < 0 else 'L'
    return out

def lat_of(h, tag=''):
    geo = side_from_corner_x(h) if RULES['lat'] == 'corner_x' else side_from_geometry(h)
    d, n_tag, n_geo, n_none, n_disagree = ({}, 0, 0, 0, 0)
    for st, g in h.groupby('StudyInstanceUID'):
        v = [str(x).strip().upper() for x in g['Laterality'].dropna()]
        if RULES['lat'] == 'corner_x' and 'ImageLaterality' in g.columns:
            v += [str(x).strip().upper() for x in g['ImageLaterality'].dropna()]
        v = [x[0] for x in v if x and x[0] in ('L', 'R')]
        side = v[0] if v else None
        if side is not None:
            n_tag += 1
            if geo.get(st) is not None and geo[st] != side:
                n_disagree += 1
        else:
            side = geo.get(st)
            n_geo += side is not None
            n_none += side is None
        d[st] = side
    log(f'{tag}laterality: {n_tag} from the tag, {n_geo} from geometry, {n_none} unresolved; tag and geometry disagree on {n_disagree} ({n_disagree / max(n_tag, 1):.1%} of the tagged)')
    return d

def probe(item):
    split, study, series, path = item
    row = {'split': split, 'StudyInstanceUID': study, 'SeriesInstanceUID': series, 'dir': path}
    try:
        files = sorted((e.name for e in os.scandir(path) if e.name.endswith('.dcm')))
        row['files'] = files
        row['n_slices'] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]), stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == 'MultiValue':
                row[t] = '|'.join((str(x) for x in v))
            else:
                row[t] = str(v)
    except Exception as exc:
        row['err'] = str(exc)[:120]
    return row

def walk(split):
    base = ROOT / split
    items = []
    if not base.is_dir():
        return pd.DataFrame(columns=['split', 'StudyInstanceUID', 'SeriesInstanceUID', 'dir', 'files', 'n_slices'] + HDR_TAGS)
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe, items))
    return pd.DataFrame(rows)

def annotate(df):
    desc = df['SeriesDescription'].fillna('') + ' ' + df['SequenceName'].fillna('')
    desc = desc.str.lower().str.replace(_SEP, ' ', regex=True)
    opts = df['ScanOptions'].fillna('').str.upper().str.split('|')
    opts_fs = opts.apply(lambda ts: any((t.strip() in FATSAT_OPTS for t in ts)))
    df['fatsat'] = desc.str.contains(_FATSAT_RX) | opts_fs
    tr = pd.to_numeric(df['RepetitionTime'], errors='coerce')
    te = pd.to_numeric(df['EchoTime'], errors='coerce')
    gre = df['ScanningSequence'].fillna('').str.upper().str.contains('GR')
    t1, t2, pdw = (desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX))
    df['weight'] = np.where(t1 & ~t2 & ~pdw, 'T1', np.where(t2 & ~pdw, 'T2', np.where(pdw, 'PD', np.where(gre, 'GRE', np.where(tr < 800, 'T1', np.where(te > 60, 'T2', np.where(tr >= 800, 'PD', 'UNK')))))))
    df['fluid'] = np.isin(df['weight'], ['PD', 'T2'])
    df['px'] = pd.to_numeric(df['PixelSpacing'].fillna('').str.split('|').str[0].replace('', np.nan), errors='coerce')
    return df

def pick_slots(series_df, plane_map):
    series_df = series_df.copy()
    series_df['plane'] = series_df['SeriesInstanceUID'].map(plane_map)
    out = {}
    for study, g in series_df.groupby('StudyInstanceUID'):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            sel = (g['plane'] == plane) & (g['fatsat'] == fs)
            if fluid is not None:
                sel &= g['fluid'] == fluid
            cand = g[sel]
            if len(cand) == 0 and RULES['slot_fallback'] and (fluid is False):
                cand = g[(g['plane'] == plane) & ~g['fatsat']]
            if len(cand):
                chosen[name] = cand.sort_values('n_slices', ascending=False).iloc[0]
        out[study] = chosen
    return out
ORDER_TAGS = [(32, 50), (32, 55), (32, 19)]
DECODE_FAILED = []

def _natural_key(name):
    return tuple((int(x) if x.isdigit() else x.lower() for x in re.split('(\\d+)', str(name))))

def _order_dominant_axis(rec):
    files, d = (rec['files'], rec['dir'])
    rows = []
    for pos, f in enumerate(files):
        ipp = inst = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True, specific_tags=['ImagePositionPatient', 'InstanceNumber'])
            raw = getattr(ds, 'ImagePositionPatient', None)
            if raw is not None and len(raw) >= 3:
                c = np.asarray(raw[:3], dtype=np.float64)
                if np.isfinite(c).all():
                    ipp = c
            n = getattr(ds, 'InstanceNumber', None)
            if n is not None:
                inst = float(n)
        except Exception:
            pass
        rows.append((f, ipp, inst, pos))
    placed = [r for r in rows if r[1] is not None]
    need = max(2, int(0.8 * len(rows)))
    if len(placed) >= need:
        xyz = np.stack([r[1] for r in placed])
        axis = int(np.argmax(np.ptp(xyz, axis=0)))
        spare = float(np.nanmedian(xyz[:, axis]))
        rows.sort(key=lambda r: (float(r[1][axis]) if r[1] is not None else spare, r[2] if r[2] is not None else float('inf'), r[3]))
    elif sum((r[2] is not None for r in rows)) >= need:
        rows.sort(key=lambda r: (r[2] if r[2] is not None else float('inf'), r[3]))
    else:
        rows.sort(key=lambda r: _natural_key(r[0]))
    return ([r[0] for r in rows], True)

def order_slices(rec):
    if RULES['order'] == 'dominant_axis':
        return _order_dominant_axis(rec)
    files, d = (rec['files'], rec['dir'])
    keyed = []
    for f in files:
        k = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True, specific_tags=ORDER_TAGS)
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            k = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
        except Exception:
            try:
                k = float(ds.InstanceNumber)
            except Exception:
                k = None
        keyed.append((k, f))
    if any((k is None for k, _ in keyed)):
        return (files, False)
    return ([f for _, f in sorted(keyed, key=lambda t: t[0])], True)

def read_slot(rec, n_slice=None, out_size=None):
    n_slice = GROUP if n_slice is None else n_slice
    out_size = IMG if out_size is None else out_size
    files, d, px = (rec.get('ordered') or rec['files'], rec['dir'], rec['px'])
    n = len(files)
    if n == 0:
        return None
    lo, hi = (int(SLICE_BAND[0] * (n - 1)), int(SLICE_BAND[1] * (n - 1)))
    idx = np.unique(np.linspace(lo, hi, n_slice).astype(int)) if hi > lo else np.array([n // 2])
    while len(idx) < n_slice:
        idx = np.append(idx, idx[-1])
    planes = []
    for i in idx[:n_slice]:
        try:
            ds = pydicom.dcmread(os.path.join(d, files[int(i)]), force=True)
            a = ds.pixel_array.astype(np.float32)
            sl = float(getattr(ds, 'RescaleSlope', 1) or 1)
            ic = float(getattr(ds, 'RescaleIntercept', 0) or 0)
            a = a * sl + ic
        except Exception:
            a = None
        planes.append(a)
    got = [k for k, p in enumerate(planes) if p is not None]
    if RULES['decode_fill'] == 'zero':
        if not got:
            DECODE_FAILED.append(rec.get('SeriesInstanceUID', d))
        planes = [np.zeros((out_size, out_size), np.float32) if p is None else p for p in planes]
        got = list(range(len(planes)))
    if not got:
        DECODE_FAILED.append(rec.get('SeriesInstanceUID', d))
        return None
    if len(got) < len(planes):
        DECODE_FAILED.append(rec.get('SeriesInstanceUID', d))
        for k, p in enumerate(planes):
            if p is None:
                planes[k] = planes[min(got, key=lambda j: abs(j - k))]
    shp = planes[0].shape
    planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]
    vol = np.stack(planes)
    if px and np.isfinite(px) and (px > 0):
        want = int(round(CROP_MM / px))
        h, w = shp
        if 16 < want < min(h, w):
            cy, cx = (h // 2, w // 2)
            half = want // 2
            vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]
    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-06), 0, 1)
    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t = F.interpolate(t, size=(out_size, out_size), mode='bilinear', align_corners=False)
    return (t.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)

def normalise_laterality(img, plane, lat):
    if lat != 'R':
        return img
    if plane in ('Coronal', 'Axial'):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])
ORDER_CACHE = os.environ.get('RSNA_ORDER_CACHE') or None

def build_cache(slot_map, plane_map, lat_map, tag):
    studies = sorted(slot_map)
    sidx = {s: i for i, s in enumerate(studies)}
    cache = np.zeros((len(studies), N_SLOT, CACHE_SLICES, IMG, IMG), np.uint8)
    mask = np.zeros((len(studies), N_SLOT), np.float32)
    log(f'{tag}: cache {cache.shape} = {cache.nbytes / 1024 ** 3:.1f} GB')
    jobs = [(st, k, plane, slot_map[st][name]) for st in studies for k, (name, plane, _, _) in enumerate(SLOTS) if name in slot_map[st]]
    n_job = len(jobs)
    t_ord = time.time()
    n_slice_total = sum((len(j[3]['files']) for j in jobs))
    log(f'{tag}: ordering {len(jobs)} slot-series ({n_slice_total} slice headers)')
    ok = done = 0
    CHUNK_O = 1024
    seen = {}
    if ORDER_CACHE and Path(ORDER_CACHE).is_file():
        try:
            import json as _json
            seen = _json.loads(Path(ORDER_CACHE).read_text())
        except (OSError, ValueError):
            seen = {}
        hit = 0
        for _, _, _, rec in jobs:
            e = seen.get(rec['SeriesInstanceUID'])
            if e and len(e['files']) == len(rec['files']):
                rec['ordered'] = e['files']
                ok += int(e['good'])
                hit += 1
        jobs = [j for j in jobs if 'ordered' not in j[3]]
        log(f'{tag}: {hit} slot-series ordered from {ORDER_CACHE}, {len(jobs)} to read')
    with ThreadPoolExecutor(max_workers=ORDER_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK_O):
            block = jobs[c0:c0 + CHUNK_O]
            for (_, _, _, rec), (files, good) in zip(block, pool.map(lambda j: order_slices(j[3]), block)):
                rec['ordered'] = files
                ok += int(good)
                done += 1
                if ORDER_CACHE:
                    seen[rec['SeriesInstanceUID']] = {'files': files, 'good': bool(good)}
            budget = min(ORDER_BUDGET_S, max(60.0, (TIME_BUDGET - (time.time() - T0)) * 0.35))
            if time.time() - t_ord > budget:
                log(f'{tag}: ordering budget spent at {done}/{len(jobs)}; the rest keep file order')
                break
    if ORDER_CACHE and done:
        import json as _json
        _t = Path(ORDER_CACHE).with_suffix('.tmp')
        _t.write_text(_json.dumps(seen))
        _t.replace(Path(ORDER_CACHE))
    log(f'{tag}: ordered {ok}/{n_job} by geometry ({n_job - ok} kept arbitrary) in {time.time() - t_ord:.0f}s')
    jobs = [(st, k, plane, slot_map[st][name]) for st in studies for k, (name, plane, _, _) in enumerate(SLOTS) if name in slot_map[st]]
    log(f'{tag}: decoding {len(jobs)} slot-series')
    n_failed_before = len(DECODE_FAILED)
    CHUNK = 512
    done = 0
    with ThreadPoolExecutor(max_workers=PIX_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK):
            block = jobs[c0:c0 + CHUNK]
            for (st, k, plane, _), img in zip(block, pool.map(lambda j: read_slot(j[3], CACHE_SLICES, IMG), block)):
                done += 1
                if img is None:
                    continue
                cache[sidx[st], k] = normalise_laterality(img, plane, lat_map.get(st)).numpy()
                mask[sidx[st], k] = 1.0
            if done % 4096 < CHUNK:
                log(f'  {tag} {done}/{len(jobs)}')
            if time.time() - T0 > TIME_BUDGET:
                log(f'  {tag}: time budget reached during decode')
                break
    n_failed = len(DECODE_FAILED) - n_failed_before
    log(f'{tag}: {int(mask.sum())}/{len(jobs)} slots filled' + (f'; {n_failed} series had a slice that would not decode' if n_failed else ''))
    gc.collect()
    return (studies, cache, mask)

class SlotHead(nn.Module):

    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2, prior=False):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden
        p_ = torch.zeros(n_out, n_slot)
        if prior and n_slot == len(SLOTS) and (n_out == len(TARGETS)):
            for t, slots in SLOT_PRIOR_TABLE.items():
                if t in TARGETS:
                    p_[TARGETS.index(t), list(slots)] = SLOT_PRIOR_STRENGTH
        self.prior = prior
        if prior:
            self.register_buffer('slot_prior', p_)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        att = torch.einsum('bsh,oh->bos', h, self.query) / self.hidden ** 0.5
        if self.prior:
            att = att + self.slot_prior.unsqueeze(0)
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -10000.0).softmax(-1)
        ctx = self.drop(torch.einsum('bos,bsh->boh', att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias

class Model(nn.Module):

    def __init__(self, backbone, dim, pool='cls_mean', prior=False):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.head = SlotHead(dim * POOL_PARTS[pool], N_SLOT, len(TARGETS), prior=prior)
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, imgs, mask, img_size=None):
        B, S = imgs.shape[:2]
        x = imgs.reshape(B * S, *imgs.shape[2:]).float().div_(255.0)
        if img_size is not None and img_size != x.shape[-1]:
            x = F.interpolate(x, size=(img_size, img_size), mode='bilinear', align_corners=False)
        x = (x - self.mean) / self.std
        out = self.backbone(pixel_values=x).last_hidden_state
        patch = out[:, 1:]
        parts = [out[:, 0], patch.mean(1)]
        if self.pool == 'cls_mean_focal':
            k = max(1, patch.shape[1] // 8)
            parts.append(patch.topk(k, dim=1).values.mean(1))
        feat = torch.cat(parts, dim=1).reshape(B, S, -1)
        return self.head(feat, mask)

def build_model(unfreeze_last, source=None, variant='small', pool='cls_mean', prior=False):
    from transformers import AutoModel
    p = source if source is not None else find_dinov2(variant)
    if p is None:
        raise FileNotFoundError('DINOv2 weights not attached')
    bb = AutoModel.from_pretrained(str(p))
    n_layer = len(bb.encoder.layer)
    for prm in bb.parameters():
        prm.requires_grad = False
    for blk in bb.encoder.layer[max(0, n_layer - unfreeze_last):]:
        for prm in blk.parameters():
            prm.requires_grad = True
    for prm in bb.layernorm.parameters():
        prm.requires_grad = True
    dim = bb.config.hidden_size
    trainable = sum((p.numel() for p in bb.parameters() if p.requires_grad))
    log(f'backbone: {n_layer} blocks, last {unfreeze_last} trainable ({trainable / 1000000.0:.1f}M params), feature dim {dim * POOL_PARTS[pool]}')
    return Model(bb, dim, pool=pool, prior=prior)
FINGERPRINT_TOL = 0.002

def fingerprint(model, dev, img_size, n_slot=None, group=None, seed=None):
    n_slot = N_SLOT if n_slot is None else n_slot
    group = GROUP if group is None else group
    seed = SEED if seed is None else seed
    g = torch.Generator().manual_seed(seed)
    imgs = torch.randint(0, 256, (2, n_slot, group, img_size, img_size), generator=g, dtype=torch.uint8).to(dev)
    mask = torch.ones(2, n_slot, device=dev)
    mask[1, -1] = 0.0
    was_training = model.training
    model.eval()
    with torch.no_grad():
        out = model(imgs, mask, img_size).float().cpu().numpy()
    if was_training:
        model.train()
    return out

def check_fingerprint(model, dev, img_size, expected, tol=FINGERPRINT_TOL, tag=''):
    got = fingerprint(model, dev, img_size)
    exp = np.asarray(expected, np.float32)
    if got.shape != exp.shape:
        raise WeightsError(f'{tag}fingerprint shape {got.shape} != stored {exp.shape}: the architecture is not the one these weights were fitted to')
    d = float(np.abs(got - exp).max())
    if d > tol:
        raise WeightsError(f'{tag}fingerprint differs by {d:.4g} (tolerance {tol:g}). The weights load but do not compute what they computed when fitted - preprocessing, resolution or architecture has moved between the two runs.')
    log(f'{tag}fingerprint matches within {d:.2g}')
    return d

class WeightsError(RuntimeError):
    pass
TTA_OVERLAP = True
TTA_POOL = 'prob'
PUBLIC_FRONTIER_TARGET_POOL = {'Fracture': 'max', 'Contusion': 'max', 'Medial Meniscus': 'max', 'Lateral Meniscus': 'max', 'ACL': 'top2', 'MCL': 'top2', "Baker's": 'max'}
TTA_TARGET_POOL = {**PUBLIC_FRONTIER_TARGET_POOL, 'Synovitis': 'original_mean'}
LEGACY_FOLD_SOFTPOOL_BETA = {'ACL': 6.0, 'MCL': 6.0, 'Medial Meniscus': 8.0, 'Lateral Meniscus': 8.0, "Baker's": 8.0, 'Contusion': 8.0, 'Fracture': 10.0}
LEGACY_FOLD_SOFTPOOL_ALPHA = {'ACL': 0.2, 'MCL': 0.2, 'Medial Meniscus': 0.25, 'Lateral Meniscus': 0.25, "Baker's": 0.2, 'Contusion': 0.2, 'Fracture': 0.15}

def window_starts(n_slice, group, overlap=None):
    overlap = TTA_OVERLAP if overlap is None else overlap
    if overlap and n_slice >= group:
        return list(range(n_slice - group + 1))
    return [g * group for g in range(max(n_slice // group, 1))]

def apply_target_window_pool(values, probs, logits, original_probs, mapping, target_idx):
    for target, mode in mapping.items():
        j = target_idx[target]
        if mode == 'max':
            values[:, j] = probs[:, :, j].max(0).values
        elif mode == 'mean':
            values[:, j] = probs[:, :, j].mean(0)
        elif mode == 'logit_mean':
            values[:, j] = torch.sigmoid(logits[:, :, j].mean(0))
        elif mode == 'original_mean':
            values[:, j] = original_probs[:, :, j].mean(0)
        elif mode in ('top2', 'top3'):
            k = min(int(mode[3:]), probs.shape[0])
            values[:, j] = probs[:, :, j].topk(k, dim=0).values.mean(0)
        else:
            raise ValueError(f'unknown TTA pooling mode for {target}: {mode}')
    return values

def legacy_fold_soft_window_pool(original_probs, target_idx):
    values = original_probs.mean(0).clone()
    for target, beta in LEGACY_FOLD_SOFTPOOL_BETA.items():
        j = target_idx[target]
        x = original_probs[:, :, j]
        weight = torch.softmax(float(beta) * x, dim=0)
        values[:, j] = (weight * x).sum(0)
    return values

@torch.no_grad()
def predict_member(model, cache, mask, idx, dev, img_size, group=None, pool=None, starts=None, jitter=False, jitter_seed=SEED, return_public_frontier=False):
    group = GROUP if group is None else group
    pool = TTA_POOL if pool is None else pool
    starts = window_starts(cache.shape[2], group) if starts is None else list(starts)
    if not starts:
        raise ValueError('predict_member was given no windows to average over')
    target_idx = {t: j for j, t in enumerate(TARGETS)}
    unknown = (set(TTA_TARGET_POOL) | set(PUBLIC_FRONTIER_TARGET_POOL)) - set(target_idx)
    if unknown:
        raise ValueError(f'unknown target(s) in TTA_TARGET_POOL: {unknown}')
    jitter_gen = torch.Generator(device=dev)
    jitter_gen.manual_seed(int(jitter_seed) % (2 ** 63 - 1))
    model.eval()
    out, public_frontier_out, public_soft_out = ([], [], [])
    for b in range(0, len(idx), EVAL_BATCH):
        sel = idx[b:b + EVAL_BATCH]
        m = torch.from_numpy(mask[sel]).to(dev)
        win_probs, win_logits, win_original_probs = ([], [], [])
        for st in starts:
            rows = torch.from_numpy(np.ascontiguousarray(cache[sel, :, st:st + group])).to(dev)
            views = [rows] + ([augment(rows, generator=jitter_gen)] if jitter else [])
            view_probs, view_logits = ([], [])
            for view in views:
                with torch.autocast('cuda', enabled=dev.type == 'cuda'):
                    z = model(view, m, img_size).float()
                view_logits.append(z)
                view_probs.append(torch.sigmoid(z))
            win_logits.append(torch.stack(view_logits).mean(0))
            win_probs.append(torch.stack(view_probs).mean(0))
            win_original_probs.append(view_probs[0])
        probs = torch.stack(win_probs)
        logits = torch.stack(win_logits)
        original_probs = torch.stack(win_original_probs)
        v = torch.sigmoid(logits.mean(0)) if pool == 'logit' else probs.mean(0)
        v = apply_target_window_pool(v, probs, logits, original_probs, TTA_TARGET_POOL, target_idx)
        out.append(v.cpu().numpy())
        if return_public_frontier:
            public_v = apply_target_window_pool(original_probs.mean(0), original_probs, logits, original_probs, PUBLIC_FRONTIER_TARGET_POOL, target_idx)
            public_frontier_out.append(public_v.cpu().numpy())
            public_soft = legacy_fold_soft_window_pool(original_probs, target_idx)
            public_soft_out.append(public_soft.cpu().numpy())
    primary = np.concatenate(out) if out else np.zeros((0, len(TARGETS)), np.float32)
    if not return_public_frontier:
        return primary
    public_frontier = np.concatenate(public_frontier_out) if public_frontier_out else np.zeros((0, len(TARGETS)), np.float32)
    public_soft = np.concatenate(public_soft_out) if public_soft_out else np.zeros((0, len(TARGETS)), np.float32)
    return (primary, public_frontier, public_soft)
BUILD_LOCK = threading.Lock()
STATE_LOCK = threading.Lock()

def _run_member(path, m, dev, Cte, Mte, idx, starts, jitter):
    t0 = time.time()
    with BUILD_LOCK:
        if 'state' in m:
            state, fp = (m['state'], None)
        else:
            ck = torch.load(Path(path) / m['file'], map_location='cpu', weights_only=False)
            state, fp = (ck['model'], ck.get('fingerprint'))
        model = build_model(int(m['config']['unfreeze_last']), variant=m['config']['variant'], pool=m['config'].get('pool', 'cls_mean'), prior=bool(m['config'].get('prior', False))).to(dev)
        model.load_state_dict(state)
        if fp is not None:
            check_fingerprint(model, dev, IMG, fp, tag=f"{m['id']}: ")
        else:
            log(f"  {m['id']}: no stored fingerprint (legacy bundle) -- accepted at reduced weight")
    t_ready = time.time()
    jitter_seed = SEED + int(hashlib.sha256(str(m['id']).encode()).hexdigest()[:8], 16)
    public_member = 'state' not in m
    predicted = predict_member(model, Cte, Mte, idx, dev, IMG, starts=starts, jitter=jitter, jitter_seed=jitter_seed, return_public_frontier=public_member)
    if public_member:
        p, public_p, public_soft = predicted
    else:
        p, public_p, public_soft = (predicted, None, None)
    t_done = time.time()
    del model, state
    gc.collect()
    if dev.type == 'cuda':
        with torch.cuda.device(dev):
            torch.cuda.empty_cache()
    passes = len(starts) * (2 if jitter else 1)
    return (p, public_p, public_soft, (t_ready - t0, (t_done - t_ready) / max(passes, 1)))

def _combine(per_member):
    all_ids = sorted({s for m in per_member for s in m['ids']})
    pos = {s: i for i, s in enumerate(all_ids)}
    acc = np.zeros((len(all_ids), len(TARGETS)), np.float64)
    tot = np.zeros(len(TARGETS), np.float64)
    for m in per_member:
        target_weight = m.get('target_weight')
        w = np.asarray(target_weight if target_weight is not None else [float(m.get('weight', 1.0))] * len(TARGETS), dtype=np.float64)
        if w.shape != (len(TARGETS),) or np.any(w < 0):
            raise ValueError(f"invalid target weights for {m.get('id')}: {w}")
        r = pd.DataFrame(m['pred']).rank(pct=True).to_numpy()
        acc[[pos[s] for s in m['ids']]] += r * w[None, :]
        tot += w
    if np.any(tot <= 0):
        raise ValueError(f'at least one target has no ensemble vote: {tot}')
    return (all_ids, acc / tot[None, :])

def combine_public_members_by_fold(per_member, pred_key='pred'):
    all_ids = sorted({study for member in per_member for study in member['ids']})
    position = {study: i for i, study in enumerate(all_ids)}
    groups = {}
    for i, member in enumerate(per_member):
        fold = member.get('fold')
        key = f'fold_{fold}' if fold is not None else f'member_{i}'
        groups.setdefault(key, []).append(member)
    fold_ranks, diagnostics = ([], [])
    for key, members_in_fold in sorted(groups.items()):
        matrices = []
        for member in members_in_fold:
            values = np.full((len(all_ids), len(TARGETS)), np.nan, np.float64)
            values[[position[study] for study in member['ids']]] = np.asarray(member[pred_key], np.float64)
            if np.isnan(values).any():
                raise WeightsError(f"{member.get('id')}: incomplete {pred_key} coverage")
            matrices.append(values)
        raw_fold_mean = np.mean(matrices, axis=0)
        fold_ranks.append(pd.DataFrame(raw_fold_mean).rank(method='average', pct=True).to_numpy(np.float64))
        diagnostics.append({'ensemble_group': key, 'members': len(members_in_fold)})
    if len(fold_ranks) != 5:
        raise WeightsError(f'legacy branch requires five folds, found {len(fold_ranks)}')
    return (all_ids, np.mean(fold_ranks, axis=0), pd.DataFrame(diagnostics))

def blend_legacy_frontier_and_soft(frontier_rank, soft_rank):
    output = np.asarray(frontier_rank, np.float64).copy()
    for j, target in enumerate(TARGETS):
        alpha = float(LEGACY_FOLD_SOFTPOOL_ALPHA.get(target, 0.0))
        if alpha:
            output[:, j] = (1.0 - alpha) * frontier_rank[:, j] + alpha * soft_rank[:, j]
    return output

def infer_from_package(path, dev=None):
    man = json.loads((Path(path) / 'manifest.json').read_text())
    members = man['members']
    log(f'weights package: {len(members)} member(s) from {path}; {len(DEVS)} device(s)')
    test_df = pd.read_csv(ROOT / 'test.csv')
    test_series = pd.read_csv(ROOT / 'test_series.csv')
    plane_map = dict(zip(test_series['SeriesInstanceUID'], test_series['Anatomical_Plane']))
    hte = annotate(walk('test_series'))
    log(f'test header pass: {len(hte)} series')
    groups = {}
    for m in members:
        groups.setdefault(m['pixel_group'], []).append(m)
    groups.update(legacy_group_members())
    per_member, public_frontier_members = ([], [])
    est = {'fixed': None, 'win': None}

    def bank(m, ids, pred, starts, jitter, public_pred=None, public_soft=None):
        if float(np.std(pred)) < 1e-09:
            log(f"  {m['id']}: degenerate predictions; not banked")
            return
        with STATE_LOCK:
            per_member.append({'id': m['id'], 'fold': m.get('fold'), 'ids': ids, 'pred': pred, 'weight': m.get('weight', 1.0), 'target_weight': m.get('target_weight'), 'holdout': m.get('holdout')})
            if public_pred is not None and len(starts) == len(starts_full):
                if float(np.std(public_pred)) < 1e-09:
                    raise WeightsError(f"{m['id']}: degenerate public-frontier prediction")
                public_frontier_members.append({'id': m['id'], 'fold': m.get('fold'), 'ids': ids, 'pred': public_pred, 'soft_pred': public_soft})
            elif public_pred is not None:
                log(f"  {m['id']}: public-frontier vote omitted because only {len(starts)} / {len(starts_full)} windows completed")
            all_ids, acc = _combine(per_member)
            write_submission(acc, all_ids, test_df, 'submission.csv')
            log(f"  banked {m['id']} fold {m.get('fold', '?')} ({len(starts)} window(s){(', jitter' if jitter else '')}); submission.csv = weighted rank mean of {len(per_member)} member(s)")
    for gi, (key, gm) in enumerate(groups.items(), 1):
        cfg = json.loads(key)
        adopt_config_globals(cfg)
        log(f"decode group {gi}/{len(groups)}: {cfg['img']}px x {cfg['slices']} slices, crop {cfg['crop_mm']} mm -> {len(gm)} member(s)")
        st_te, Cte, Mte = build_cache(pick_slots(hte, plane_map), plane_map, lat_of(hte, 'test '), f'test g{gi}')
        idx = np.arange(len(st_te))
        starts_full = window_starts(Cte.shape[2], GROUP)
        pending = sorted(gm, key=lambda m: -(m.get('holdout') or 0))
        left_after = sum((len(g) for j, (_, g) in enumerate(groups.items(), 1) if j > gi))

        def pop_next():
            with STATE_LOCK:
                if not pending:
                    return (None, None, False)
                left = TIME_BUDGET - (time.time() - T0)
                remaining = len(pending) + left_after
                slots_left = -(-remaining // len(DEVS))
                starts, jit = (starts_full, False)
                if est['fixed'] is not None and est['win'] is not None:
                    afford = max(left * 0.9, 0.0)
                    room = afford / max(slots_left, 1)
                    if est['fixed'] + est['win'] > room:
                        log(f'  {left / 60:.0f} min left: surrendering {len(pending)} member(s); not one more fits')
                        pending.clear()
                        return (None, None, False)
                    jit = est['fixed'] + 2 * len(starts_full) * est['win'] <= room * 0.6
                    per_win = est['win'] * (2 if jit else 1)
                    n_win = int((room - est['fixed']) / per_win) if per_win > 0 else len(starts_full)
                    n_win = max(1, min(len(starts_full), n_win))
                    if n_win < len(starts_full):
                        mid = (len(starts_full) - n_win) // 2
                        starts = starts_full[mid:mid + n_win]
                return (pending.pop(0), starts, jit)

        def worker(dev):
            others = [d for d in DEVS if d is not dev]
            while True:
                m, starts, jit = pop_next()
                if m is None:
                    return
                for attempt, d in enumerate([dev] + others[:1]):
                    try:
                        p, public_p, public_soft, (fs, ws) = _run_member(path, m, d, Cte, Mte, idx, starts, jit)
                        with STATE_LOCK:
                            est['fixed'], est['win'] = (fs, ws)
                        bank(m, st_te, p, starts, jit, public_p, public_soft)
                        break
                    except Exception as exc:
                        log(f"  MEMBER {m['id']} failed on {d} ({type(exc).__name__}: {exc}); " + ('retrying on peer device' if attempt == 0 and others else 'dropped -- costs one vote, not the run'))
                        if d.type == 'cuda':
                            with torch.cuda.device(d):
                                torch.cuda.empty_cache()
        threads = [threading.Thread(target=worker, args=(d,)) for d in DEVS]
        for t in threads:
            t.start()
        for t in threads:
            t.join()
        del Cte, Mte
        gc.collect()
    if not per_member:
        raise WeightsError('no member produced predictions; submission stays at 0.5')
    all_ids, acc = _combine(per_member)
    sub = write_submission(acc, all_ids, test_df, 'submission.csv')
    log(f'final submission.csv = weighted rank mean of {len(per_member)} member(s); {sub.shape}; nulls {int(sub[TARGETS].isna().sum().sum())}')
    if len(public_frontier_members) == len(members):
        frontier_ids, frontier_acc = _combine(public_frontier_members)
        frontier_sub = write_submission(frontier_acc, frontier_ids, test_df, 'submission_public_0899.csv')
        log(f'submission_public_0899.csv = exact no-jitter public-frontier rank mean of {len(public_frontier_members)} member(s); {frontier_sub.shape}; nulls {int(frontier_sub[TARGETS].isna().sum().sum())}')
        fold_ids, fold_frontier, fold_diagnostics = combine_public_members_by_fold(public_frontier_members, 'pred')
        soft_ids, fold_soft, _ = combine_public_members_by_fold(public_frontier_members, 'soft_pred')
        if fold_ids != soft_ids:
            raise WeightsError('legacy hard/soft study order mismatch')
        legacy_prediction = blend_legacy_frontier_and_soft(fold_frontier, fold_soft)
        legacy_sub = write_submission(legacy_prediction, fold_ids, test_df, 'submission_legacy_fold_blend.csv')
        fold_diagnostics.to_csv('legacy_fold_diagnostics.csv', index=False)
        log(f'legacy DINO aggregation written from five folds; {legacy_sub.shape}')
    else:
        log(f'public-frontier fallback not emitted: {len(public_frontier_members)} / {len(members)} required public members completed')
    return sub

def adopt_config_globals(cfg):
    global IMG, CACHE_IMG, GROUP, CACHE_SLICES, N_GROUP, CROP_MM, SLICE_BAND, RULES
    CACHE_IMG = IMG = int(cfg['img'])
    GROUP = int(cfg['group'])
    CACHE_SLICES = int(cfg['slices'])
    N_GROUP = max(CACHE_SLICES // GROUP, 1)
    CROP_MM = float(cfg['crop_mm'])
    SLICE_BAND = tuple((float(x) for x in cfg['band']))
    rules = cfg.get('rules') or RULES_NATIVE
    unknown = {k: v for k, v in rules.items() if k not in RULES_NATIVE or v not in (RULES_NATIVE[k], RULES_LEGACY[k])}
    if unknown:
        raise WeightsError(f'the members record pixel rules this pipeline cannot reproduce: {unknown}')
    RULES = {**RULES_NATIVE, **rules}
    if [s[0] for s in SLOTS] != list(cfg['slots']):
        raise WeightsError(f"the members were fitted on slots {cfg['slots']} and this pipeline defines {[s[0] for s in SLOTS]}; a weight would be read against the wrong slot")

def augment(imgs, generator=None):
    lead = imgs.shape[:-3]
    x = imgs.reshape(-1, *imgs.shape[-3:]).float()
    n, dev = (x.shape[0], x.device)
    rot = (torch.rand(n, device=dev, generator=generator) - 0.5) * 2 * (AUG_ROT_DEG * np.pi / 180)
    sc = 1.0 + torch.rand(n, device=dev, generator=generator) * AUG_SCALE
    tx = (torch.rand(n, device=dev, generator=generator) - 0.5) * 2 * AUG_SHIFT
    ty = (torch.rand(n, device=dev, generator=generator) - 0.5) * 2 * AUG_SHIFT
    cos, sin = (torch.cos(rot) / sc, torch.sin(rot) / sc)
    theta = torch.zeros(n, 2, 3, device=dev, dtype=torch.float32)
    theta[:, 0, 0], theta[:, 0, 1], theta[:, 0, 2] = (cos, -sin, tx)
    theta[:, 1, 0], theta[:, 1, 1], theta[:, 1, 2] = (sin, cos, ty)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=False)
    scale = 1.0 + (torch.rand(n, 1, 1, 1, device=dev, generator=generator) - 0.5) * 2 * AUG_INTENSITY
    x = (x * scale).clamp(0, 255)
    return x.reshape(*lead, *x.shape[-3:]).to(imgs.dtype)

def write_submission(pred, studies, test_df, path):
    sub = pd.DataFrame(pd.DataFrame(pred).rank(pct=True).values, columns=TARGETS)
    sub.insert(0, 'StudyInstanceUID', studies)
    sub = test_df[['StudyInstanceUID']].merge(sub, on='StudyInstanceUID', how='left')
    sub[TARGETS] = sub[TARGETS].fillna(0.5)
    sub.to_csv(path, index=False)
    return sub

def find_dinov2(variant='small'):
    if not (DINO / 'config.json').is_file():
        raise FileNotFoundError(DINO)
    return DINO

def legacy_group_members():
    return {}

def run_dinov2():
    global V8_D2_WEIGHTED_FRAME, V8_D2_SOFT_FRAME, V8_D2_PUBLIC_FRAME

    path = ASSET / 'rsna-knee-weights'
    infer_from_package(path, DEVS[0])

    work = Path('/kaggle/working')
    weighted_path = work / 'submission.csv'
    public_path = work / 'submission_public_0899.csv'
    soft_path = work / 'submission_legacy_fold_blend.csv'

    if not public_path.is_file():
        raise RuntimeError('public DINOv2 frontier was not produced')

    V8_D2_WEIGHTED_FRAME = pd.read_csv(
        weighted_path,
        dtype={'StudyInstanceUID': str},
    )
    V8_D2_PUBLIC_FRAME = pd.read_csv(
        public_path,
        dtype={'StudyInstanceUID': str},
    )

    if soft_path.is_file():
        V8_D2_SOFT_FRAME = pd.read_csv(
            soft_path,
            dtype={'StudyInstanceUID': str},
        )
    else:
        V8_D2_SOFT_FRAME = V8_D2_PUBLIC_FRAME.copy()

    public_path.replace(weighted_path)

    for name in (
        'submission_legacy_fold_blend.csv',
        'legacy_fold_diagnostics.csv',
    ):
        candidate = work / name
        if candidate.is_file():
            candidate.unlink()

run_dinov2()


### 【解説】セル2: 独自アーキテクチャ系（A5 / Net）— 「スライスの束」をどう1つにまとめるか

**何をしているか**: 別のバックボーン（timm系）で特徴を抽出し、**スライス方向・スロット方向の集約（プーリング）**を
何種類も実装して組み合わせ、2本目の予測を作ります。最後にセル1の結果と**順位で合成**して `submission.csv` を書きます。

**なぜそうするのか（このセルの本質はプーリング設計）**:

MRIは1シリーズが**十数〜数十枚のスライスの束**です。異常はそのうち数枚にしか写りません。
「束をどう1つのベクトルにまとめるか」がこのタスクの核心で、このセルにはその答えが何通りも入っています。

| クラス | 集約の考え方 | 効くところ |
|---|---|---|
| `MeanMaxPool` | 平均と最大を連結 | 平均＝全体の傾向、最大＝一番目立つスライス。両方残す最も基本的な保険 |
| `LabelAttentionPool` | **ラベルごとに**注目するスライスを学習 | ACLは矢状断中央、半月板は別スライス——**異常ごとに見るべき場所が違う**ことを表現できる |
| `TokenXAttnPool` / `_GatedDelta` | クエリを立ててトークン集合に注意 | スライス間の関係（隣とどう違うか）を捉える |
| `SlotDepthMixer` / `DepthCompress` | 深さ（スライス）方向を先に圧縮 | 計算量を落としつつ、深さ方向の連続性を保つ |
| `ClsAddPool` / `TokenResidualPool` | CLSトークンに残差を足す | 事前学習済み表現を壊さずに微修正する |

**★ `LabelAttentionPool` が一番重要**です。単純な平均プーリングだと、
「40枚中2枚にしか写っていない骨折」の信号が38枚のノイズに薄められて消えます。
ラベル別のアテンションは「このラベルについてはこのスライスを見ろ」を**データから学習**するので、
**稀で局所的な所見の感度が上がります**。マクロ平均AUCは稀ラベルも対等に扱うため、
ここの改善がスコアに直結します。

**その他の読みどころ**:
- `segment_softmax` / `_seg_mean_max` / `_pad_kv`: 検査ごとにスライス枚数が違うので、
  **可変長のまとまり（セグメント）をバッチにパディングし、マスクで無効部分を除外**しています。
  `torch.scatter_reduce` を使った実装は、可変長データを扱う際の実務的な書き方として参考になります。
- `amp_for` / bfloat16: GPUの世代を見て混合精度の型を選びます。T4（sm_75）は bf16 非対応なので fp16 に落とす、
  という分岐です。**ハードウェアに合わせて数値精度を選ぶ**のは推論時間を守るための現実的な工夫。
- `SLICE_BAND = (0.12, 0.88)`: 端のスライス（体外の空気や打ち切り）を捨てます。
  セル1では `(0.2, 0.8)` とより保守的で、**同じ設計思想でもパイプラインごとに値を変えて多様性にしている**点に注目。
- **最後の合成**: `_a5_base_rank` と `_a5_ours_rank` を `A5_W` で加重平均。
  両方とも `rank(pct=True)` で順位化済みです。**確率の平均ではなく順位の平均**——
  AUCが順位しか見ない以上、これが最も素直な合成です。


In [ ]:
_A5_SAVED = dict(globals())
import gc, os, time, warnings
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import pydicom
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings('ignore')
cv2.setNumThreads(1)
CROP_MM = 130.0
SIZE = 336
SLICE_BAND = (0.12, 0.88)
N_SLICE = 16
INTENSITY = 'slice'
SLOTS = [('Sagittal', 1), ('Sagittal', 0), ('Coronal', 1), ('Coronal', 0), ('Axial', 1), ('Axial', 0)]
N_SLOT = len(SLOTS)
LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
COMP = Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')
CKPT = ASSET / 'knee-mri-fold-weights'
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'competition : {COMP}')
print(f'checkpoints : {CKPT}')
print(f'device      : {DEV}')
for i in range(torch.cuda.device_count() if DEV == 'cuda' else 0):
    cc = torch.cuda.get_device_capability(i)
    print(f'  gpu{i}       : {torch.cuda.get_device_name(i)} sm_{cc[0]}{cc[1]}, {torch.cuda.get_device_properties(i).total_memory / 2 ** 30:.0f} GiB, native bf16={cc >= (8, 0)}')
SERIES_ROOT = COMP / 'test_series'
if not SERIES_ROOT.exists():
    SERIES_ROOT = COMP / 'train_series'
print('series root:', SERIES_ROOT)

def ordered_files(sdir, cap=64):
    keyed = []
    for f in sdir.glob('*.dcm'):
        try:
            ds = pydicom.dcmread(str(f), stop_before_pixels=True)
            keyed.append((int(ds.InstanceNumber), str(f)))
        except Exception:
            continue
        if len(keyed) >= cap * 4:
            break
    return [f for _, f in sorted(keyed)]

def series_side(path):
    try:
        return float(pydicom.dcmread(path, stop_before_pixels=True).ImagePositionPatient[0])
    except Exception:
        return 0.0

def read_crop(path):
    try:
        ds = pydicom.dcmread(path)
        arr = ds.pixel_array.astype(np.float32)
    except Exception:
        return None
    try:
        ps = float(ds.PixelSpacing[0])
    except Exception:
        ps = CROP_MM / max(arr.shape)
    half = int(round(CROP_MM / ps / 2))
    cy, cx = (arr.shape[0] // 2, arr.shape[1] // 2)
    y0, y1 = (max(0, cy - half), min(arr.shape[0], cy + half))
    x0, x1 = (max(0, cx - half), min(arr.shape[1], cx + half))
    crop = arr[y0:y1, x0:x1]
    return None if crop.size == 0 else crop

def window(crop, lo, hi, flip):
    c = np.clip((crop - lo) / max(hi - lo, 1e-06), 0, 1)
    img = cv2.resize(c, (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    return img[:, ::-1].copy() if flip else img

def render(path, flip):
    crop = read_crop(path)
    if crop is None:
        return None
    lo, hi = np.percentile(crop[::4, ::4], [1, 99])
    return window(crop, lo, hi, flip)

def build_study(args):
    idx, study, recs = args
    out = np.zeros((N_SLOT, N_SLICE, SIZE, SIZE), np.uint8)
    mask = np.zeros(N_SLOT, np.uint8)
    rows = pd.DataFrame(recs)
    if len(rows):
        for s_i, (plane, fs) in enumerate(SLOTS):
            sub = rows[(rows.Anatomical_Plane == plane) & (rows.Fat_Suppression == fs)]
            if sub.empty:
                continue
            files = ordered_files(SERIES_ROOT / study / sub.iloc[0].SeriesInstanceUID)
            if not files:
                continue
            flip = plane != 'Sagittal' and series_side(files[0]) < 0
            lo, hi = SLICE_BAND
            i0 = int(round(lo * (len(files) - 1)))
            i1 = int(round(hi * (len(files) - 1)))
            avail = list(range(i0, i1 + 1))
            if len(avail) >= N_SLICE:
                picks = [avail[int(round(t))] for t in np.linspace(0, len(avail) - 1, N_SLICE)]
                off = 0
            else:
                picks, off = (avail, (N_SLICE - len(avail)) // 2)
            if INTENSITY == 'series':
                crops = [read_crop(files[p]) for p in picks]
                got = [x for x in crops if x is not None]
                if got:
                    samp = np.concatenate([x[::4, ::4].ravel() for x in got])
                    lo_, hi_ = np.percentile(samp, [1, 99])
                    for c, x in enumerate(crops):
                        if x is None:
                            x = read_crop(files[min(len(files) - 1, picks[c] + 1)])
                        if x is not None:
                            out[s_i, off + c] = (window(x, lo_, hi_, flip) * 255).astype(np.uint8)
            else:
                for c, p in enumerate(picks):
                    img = render(files[p], flip)
                    if img is None:
                        img = render(files[min(len(files) - 1, p + 1)], flip)
                    if img is not None:
                        out[s_i, off + c] = (img * 255).astype(np.uint8)
            mask[s_i] = len(picks)
    return (idx, out, mask)
sub_df = pd.read_csv(COMP / 'sample_submission.csv')
ser_csv = pd.read_csv(COMP / 'test_series.csv')
if not (COMP / 'test_series').exists():
    ser_csv = pd.read_csv(COMP / 'train_series.csv')
ser_csv = ser_csv.loc[:, ~ser_csv.columns.duplicated()]
studies = sub_df.StudyInstanceUID.tolist()
by = {s: g.to_dict('records') for s, g in ser_csv[ser_csv.StudyInstanceUID.isin(set(studies))].groupby('StudyInstanceUID')}
print(f'{len(studies):,} test studies, {len(by):,} with series metadata')
N_SLOT_TYPES, MASK_IDX = (6, 0)

def segment_softmax(scores, sidx, B):
    T, K = scores.shape
    idx = sidx.unsqueeze(1).expand(-1, K)
    m = torch.full((B, K), float('-inf'), device=scores.device, dtype=scores.dtype)
    m = m.scatter_reduce(0, idx, scores, reduce='amax', include_self=True)
    e = (scores - m[sidx]).exp()
    s = torch.zeros(B, K, device=scores.device, dtype=scores.dtype).index_add_(0, sidx, e)
    return e / s[sidx].clamp(min=1e-06)

class MeanMaxPool(nn.Module):

    def forward(self, f, sidx, B, slot=None, return_attn=False):
        D = f.shape[1]
        cnt = torch.zeros(B, device=f.device, dtype=f.dtype).index_add_(0, sidx, torch.ones(f.shape[0], device=f.device, dtype=f.dtype))
        mean = torch.zeros(B, D, device=f.device, dtype=f.dtype).index_add_(0, sidx, f)
        mean = mean / cnt.clamp(min=1).unsqueeze(1)
        mx = torch.full((B, D), -10000.0, device=f.device, dtype=f.dtype)
        mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), f, reduce='amax', include_self=True)
        return (torch.cat([mean, mx], 1), None)

class LabelAttentionPool(nn.Module):

    def __init__(self, d, n_labels=12, n_heads=4, slot_bias=True):
        super().__init__()
        self.d, self.k, self.h = (d, n_labels, n_heads)
        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.key, self.val = (nn.Linear(d, d), nn.Linear(d, d))
        self.slot_bias = nn.Parameter(torch.zeros(n_labels, N_SLOT_TYPES + 1)) if slot_bias else None

    def forward(self, f, sidx, B, slot=None, return_attn=False):
        scores = self.key(f) @ self.q.t() / self.d ** 0.5
        if self.slot_bias is not None and slot is not None:
            scores = scores + self.slot_bias.t()[slot]
        a = segment_softmax(scores, sidx, B)
        out = torch.zeros(B, self.k, self.d, device=f.device, dtype=f.dtype)
        out = out.index_add_(0, sidx, a.unsqueeze(-1) * self.val(f).unsqueeze(1))
        return (out, a)

class TokenXAttnPool(nn.Module):

    def __init__(self, d, n_labels=12, n_heads=6, dropout=0.2):
        super().__init__()
        self.d, self.k = (d, n_labels)
        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.slot_emb = nn.Embedding(N_SLOT_TYPES + 1, d, padding_idx=0)
        self.kv_norm = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)

    def forward(self, tok, sidx, B, slot=None, return_attn=False):
        T, N, D = tok.shape
        cnt = torch.bincount(sidx, minlength=B)
        S = int(cnt.max().item())
        starts = torch.cumsum(cnt, 0) - cnt
        pos = torch.arange(T, device=tok.device) - starts[sidx]
        kv = tok + self.slot_emb(slot).unsqueeze(1)
        pad = tok.new_zeros(B, S, N, D)
        pad[sidx, pos] = kv
        keep = torch.zeros(B, S, dtype=torch.bool, device=tok.device)
        keep[sidx, pos] = True
        kpm = ~keep.repeat_interleave(N, dim=1)
        pad = self.kv_norm(pad.reshape(B, S * N, D))
        q = self.q.unsqueeze(0).expand(B, -1, -1)
        att, w = self.attn(q, pad, pad, key_padding_mask=kpm, need_weights=return_attn, average_attn_weights=True)
        cls = tok[:, 0]
        mean = torch.zeros(B, D, device=tok.device, dtype=tok.dtype).index_add_(0, sidx, cls) / cnt.clamp(min=1).unsqueeze(1)
        mx = torch.full((B, D), -10000.0, device=tok.device, dtype=tok.dtype)
        mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), cls, reduce='amax', include_self=True)
        base = torch.cat([mean, mx], 1).unsqueeze(1).expand(-1, self.k, -1)
        return (torch.cat([att, base], -1), w)

class ViTSlotToken(nn.Module):

    def __init__(self, vit, n_cat, dim=None):
        super().__init__()
        self.vit = vit
        d = dim or vit.embed_dim
        self.tok = nn.Embedding(n_cat + 1, d, padding_idx=MASK_IDX)
        self.num_features = vit.num_features
        self._orig_prefix = getattr(vit, 'num_prefix_tokens', 1)
        vit.num_prefix_tokens = self._orig_prefix + 1
        for blk in vit.blocks:
            a = getattr(blk, 'attn', None)
            if a is not None and hasattr(a, 'num_prefix_tokens'):
                a.num_prefix_tokens = a.num_prefix_tokens + 1

    @staticmethod
    def _maybe(mod, x):
        return x if mod is None else mod(x)

    def forward_features(self, x, cat):
        v = self.vit
        x = v.patch_embed(x)
        pos = v._pos_embed(x)
        rope = None
        if isinstance(pos, tuple):
            x, rope = pos
        else:
            x = pos
        x = self._maybe(getattr(v, 'patch_drop', None), x)
        x = self._maybe(getattr(v, 'norm_pre', None), x)
        npt = self._orig_prefix
        tok = self.tok(cat).unsqueeze(1)
        x = torch.cat([x[:, :npt], tok, x[:, npt:]], dim=1)
        if rope is not None:
            if getattr(v, 'rope_mixed', False):
                for i, blk in enumerate(v.blocks):
                    x = blk(x, rope=rope[i])
            else:
                for blk in v.blocks:
                    x = blk(x, rope=rope)
        else:
            x = v.blocks(x)
        return v.norm(x)

    def forward_head(self, x, pre_logits=True):
        return self.vit.forward_head(x, pre_logits=pre_logits)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

class _GatedDepthBlock(nn.Module):

    def __init__(self, n_slice, dropout=0.0, ls_init=0.1):
        super().__init__()
        self.norm = nn.GroupNorm(1, n_slice)
        self.v = nn.Conv2d(n_slice, n_slice, 1)
        self.g = nn.Conv2d(n_slice, n_slice, 1)
        self.out = nn.Conv2d(n_slice, n_slice, 1)
        self.gamma = nn.Parameter(torch.full((n_slice, 1, 1), ls_init))
        self.drop = nn.Dropout2d(dropout) if dropout else nn.Identity()

    def forward(self, x):
        z = self.norm(x)
        return x + self.gamma * self.drop(self.out(self.v(z) * F.silu(self.g(z))))

class DepthCompress(nn.Module):

    def __init__(self, n_slice=16, out_ch=3, depth=1, dropout=0.0, ls_init=0.1, imagenet=True, proj_noise=0.25):
        super().__init__()
        self.imagenet = imagenet
        self.blocks = nn.ModuleList([_GatedDepthBlock(n_slice, dropout, ls_init) for _ in range(depth)])
        self.proj = nn.Conv2d(n_slice, out_ch, 1, bias=True)
        if imagenet:
            self.register_buffer('mu', torch.tensor(IMAGENET_MEAN).view(1, -1, 1, 1))
            self.register_buffer('sd', torch.tensor(IMAGENET_STD).view(1, -1, 1, 1))

    def forward(self, x):
        keep = (x.amax(dim=1, keepdim=True) > 0).to(x.dtype)
        z = x
        for b in self.blocks:
            z = b(z)
        z = self.proj(z)
        if self.imagenet:
            z = (z - self.mu.to(z.dtype)) / self.sd.to(z.dtype)
        return z * keep
N_PLANE, N_CONTRAST = (3, 2)
_PLANE_OF = lambda s: torch.clamp(s - 1, 0, 5) // 2
_CONTRAST_OF = lambda s: torch.clamp(s - 1, 0, 5) % 2

class SlotDepthMixer(nn.Module):

    def __init__(self, n_slice=16, ksize=5, alpha_max=0.25):
        super().__init__()
        self.n_slice, self.ksize, self.r = (n_slice, ksize, ksize // 2)
        self.alpha_max = alpha_max
        b = torch.tensor([1.0, 4.0, 6.0, 4.0, 1.0])
        self.register_buffer('base', b.log()[self.r:])
        n_u = self.r + 1
        self.shared = nn.Parameter(torch.zeros(n_u))
        self.plane_k = nn.Parameter(torch.zeros(N_PLANE, n_u))
        self.contrast_k = nn.Parameter(torch.zeros(N_CONTRAST, n_u))
        self.g0 = nn.Parameter(torch.zeros(()))
        self.gate_p = nn.Parameter(torch.zeros(N_PLANE))
        self.gate_c = nn.Parameter(torch.zeros(N_CONTRAST))
        idx = torch.arange(n_slice)
        self.register_buffer('off', idx[None, :] - idx[:, None])

    def kernel(self, slot):
        p, c = (_PLANE_OF(slot), _CONTRAST_OF(slot))
        half = self.base + self.shared + self.plane_k[p] + self.contrast_k[c]
        full = torch.cat([half.flip(-1)[..., :self.r], half], dim=-1)
        return F.softmax(full, dim=-1)

    def alpha(self, slot):
        p, c = (_PLANE_OF(slot), _CONTRAST_OF(slot))
        return self.alpha_max * torch.tanh(self.g0 + self.gate_p[p] + self.gate_c[c])

    def forward(self, x, slot, vmask):
        T, S, H, W = x.shape
        if vmask is None:
            raise ValueError('stem=mixer requires the padding mask')
        k = self.kernel(slot)
        v = vmask.to(k.dtype)
        d = self.off + self.r
        inb = (d >= 0) & (d < self.ksize)
        kk = k[:, d.clamp(0, self.ksize - 1)] * inb
        M = kk * v[:, None, :]
        den = M.sum(-1, keepdim=True)
        eye = torch.eye(S, device=x.device, dtype=M.dtype).expand(T, S, S)
        ok = (den > 1e-06) & v[:, :, None].bool()
        M = torch.where(ok, M / den.clamp(min=1e-06), eye)
        a = self.alpha(slot)[:, None, None]
        Aop = ((1.0 - a) * eye + a * M).to(x.dtype)
        if x.is_contiguous(memory_format=torch.channels_last) and (not x.is_contiguous()):
            y = torch.bmm(x.permute(0, 2, 3, 1).reshape(T, H * W, S), Aop.transpose(1, 2))
            return y.reshape(T, H, W, S).permute(0, 3, 1, 2)
        return torch.bmm(Aop, x.reshape(T, S, H * W)).reshape(T, S, H, W)

def _seg_mean_max(v, sidx, B):
    D = v.shape[1]
    cnt = torch.zeros(B, device=v.device, dtype=v.dtype).index_add_(0, sidx, torch.ones(v.shape[0], device=v.device, dtype=v.dtype))
    mean = torch.zeros(B, D, device=v.device, dtype=v.dtype).index_add_(0, sidx, v)
    mean = mean / cnt.clamp(min=1).unsqueeze(1)
    mx = torch.full((B, D), -10000.0, device=v.device, dtype=v.dtype)
    mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), v, reduce='amax', include_self=True)
    return torch.cat([mean, mx], 1)

def _pad_kv(x, sidx, B, norm):
    T, P, D = x.shape
    cnt = torch.bincount(sidx, minlength=B)
    S = int(cnt.max().item())
    starts = torch.cumsum(cnt, 0) - cnt
    pos = torch.arange(T, device=x.device) - starts[sidx]
    pad = x.new_zeros(B, S, P, D)
    pad[sidx, pos] = x
    keep = torch.zeros(B, S, dtype=torch.bool, device=x.device)
    keep[sidx, pos] = True
    return (norm(pad.reshape(B, S * P, D)), ~keep.repeat_interleave(P, dim=1))

class _GatedDelta(nn.Module):

    def __init__(self, d, n_labels, n_heads, dropout):
        super().__init__()
        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.kv_norm = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.d_norm = nn.LayerNorm(d)
        self.dw = nn.Parameter(torch.randn(n_labels, d) * (1.0 / d ** 0.5))
        self.db = nn.Parameter(torch.zeros(n_labels))
        self.gate = nn.Parameter(torch.zeros(n_labels))

    def delta(self, pat, sidx, B, return_attn):
        kv, kpm = _pad_kv(pat, sidx, B, self.kv_norm)
        q = self.q.unsqueeze(0).expand(B, -1, -1)
        att, w = self.attn(q, kv, kv, key_padding_mask=kpm, need_weights=return_attn, average_attn_weights=True)
        return ((self.d_norm(att) * self.dw).sum(-1) + self.db, w)

class TokenResidualPool(_GatedDelta):

    def __init__(self, d, n_labels=12, n_heads=6, pe=64, dropout=0.2):
        super().__init__(d, n_labels, n_heads, dropout)
        self.base = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(dropout), nn.Linear(2 * d + pe, n_labels))

    def forward(self, tok, slot, sidx, B, pres, return_attn=False):
        base = self.base(torch.cat([_seg_mean_max(tok[:, 1:].mean(1), sidx, B), pres], 1))
        d_, w = self.delta(tok[:, 1:], sidx, B, return_attn)
        return (base + self.gate * d_, w)

class CodexResidualPool(_GatedDelta):

    def __init__(self, d, n_labels=12, n_heads=6, pe=64, dropout=0.2):
        super().__init__(d, n_labels, n_heads, dropout)
        self.base = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(dropout), nn.Linear(2 * d + pe, n_labels))

    def forward(self, tok, slot, sidx, B, pres, return_attn=False):
        base = self.base(torch.cat([_seg_mean_max(tok[:, 0], sidx, B), pres], 1))
        d_, w = self.delta(tok[:, 1:], sidx, B, return_attn)
        return (base + self.gate * d_, w)

class ClsAddPool(nn.Module):

    def __init__(self, d, n_labels=12, pe=64, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(nn.LayerNorm(4 * d + pe), nn.Dropout(dropout), nn.Linear(4 * d + pe, n_labels))

    def forward(self, tok, slot, sidx, B, pres, return_attn=False):
        return (self.net(torch.cat([_seg_mean_max(tok[:, 1:].mean(1), sidx, B), _seg_mean_max(tok[:, 0], sidx, B), pres], 1)), None)

class Readout(nn.Module):

    def __init__(self, pool, d, n_labels=12, pe=64):
        super().__init__()
        self.pool_kind, self.k = (pool, n_labels)
        self.pres_emb = nn.Embedding(N_SLOT_TYPES + 1, pe, padding_idx=0)
        if pool in ('xres', 'clsadd', 'xcodex'):
            self.pool = {'xres': TokenResidualPool, 'clsadd': ClsAddPool, 'xcodex': CodexResidualPool}[pool](d, n_labels, pe=pe)
        elif pool in ('attn', 'xattn'):
            if pool == 'xattn':
                self.pool = TokenXAttnPool(d, n_labels)
                wd = 3 * d + pe
            else:
                self.pool = LabelAttentionPool(d, n_labels)
                wd = d + pe
            self.norm = nn.LayerNorm(wd)
            self.w = nn.Parameter(torch.randn(n_labels, wd) * (1.0 / wd ** 0.5))
            self.b = nn.Parameter(torch.zeros(n_labels))
        else:
            self.pool = MeanMaxPool()
            self.net = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(0.2), nn.Linear(2 * d + pe, n_labels))
        self.drop = nn.Dropout(0.2)

    def forward(self, f, slot, sidx, B, return_attn=False):
        pe = self.pres_emb(slot)
        pres = torch.zeros(B, pe.shape[1], device=f.device, dtype=f.dtype).index_add_(0, sidx, pe)
        if self.pool_kind in ('xres', 'clsadd', 'xcodex'):
            return self.pool(f, slot, sidx, B, pres)[0]
        pooled, attn = self.pool(f, sidx, B, slot=slot, return_attn=return_attn)
        if self.pool_kind in ('attn', 'xattn'):
            x = torch.cat([pooled, pres.unsqueeze(1).expand(-1, self.k, -1)], -1)
            x = self.drop(self.norm(x))
            return (x * self.w).sum(-1) + self.b
        return self.net(torch.cat([pooled, pres], 1))

class Net(nn.Module):

    def __init__(self, enc, cond, n_meta=0, pool='mean_max', stem='native', n_slice=16):
        super().__init__()
        self.enc, self.cond = (enc, cond)
        self.compress = DepthCompress(n_slice, 3) if stem == 'compress' else None
        self.mixer = SlotDepthMixer(n_slice) if stem == 'mixer' else None
        self.tokens = pool in ('xattn', 'xres', 'clsadd', 'xcodex')
        D = enc.num_features
        self.meta_mlp = nn.Sequential(nn.LayerNorm(n_meta), nn.Linear(n_meta, 128), nn.GELU(), nn.Linear(128, D)) if n_meta > 0 else None
        self.readout = Readout(pool, D)
        if cond == 'post':
            self.slot_emb = nn.Embedding(N_SLOT_TYPES + 1, D, padding_idx=MASK_IDX)

    def forward(self, im, slot, smeta, sidx, B, vm=None):
        if self.mixer is not None:
            im = self.mixer(im, slot, vm)
        if self.compress is not None:
            im = self.compress(im)
        f = self.enc.forward_features(im, slot) if self.cond == 'token' else self.enc.forward_features(im)
        if self.tokens:
            inner = getattr(self.enc, 'vit', self.enc)
            orig = getattr(self.enc, '_orig_prefix', getattr(inner, 'num_prefix_tokens', 1))
            f = torch.cat([f[:, :1], f[:, orig:]], 1)
        else:
            f = self.enc.forward_head(f, pre_logits=True)
            if f.dim() > 2:
                f = f.flatten(1)
        ex = (lambda v: v.unsqueeze(1)) if self.tokens else lambda v: v
        if self.cond == 'post':
            f = f + ex(self.slot_emb(slot))
        if self.meta_mlp is not None and smeta.shape[1] > 0:
            mt = self.meta_mlp(smeta)
            f = torch.cat([f, mt.unsqueeze(1)], 1) if self.tokens else f + mt
        return self.readout(f, slot, sidx, B)
models = []
for ckpt_path in sorted(CKPT.glob('*_f*.pt')):
    z = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    cfg = z['cfg']
    _stem = cfg.get('stem', 'native')
    _in = 3 if _stem == 'compress' else cfg.get('n_slice', 16)
    enc = timm.create_model(cfg['backbone'], pretrained=False, num_classes=0, in_chans=_in, **{'img_size': cfg['img']} if 'vit_' in cfg['backbone'] else {})
    if cfg['cond'] == 'token':
        enc = ViTSlotToken(enc, N_SLOT_TYPES)
    m = Net(enc, cfg['cond'], cfg.get('n_meta', 0), cfg['pool'], stem=_stem, n_slice=cfg.get('n_slice', 16))
    missing, unexpected = m.load_state_dict(z['state_dict'], strict=False)
    assert not missing, f'missing {missing[:5]}'
    assert not unexpected, f'unexpected {unexpected[:5]}'
    models.append(m.eval())
    print(f"loaded {ckpt_path.name}  fold {z['fold']}  {cfg['backbone']} pool={cfg['pool']} meta={cfg['meta']}")
CFG = cfg
assert CFG.get('n_meta', 0) == 0, f"checkpoint expects {CFG['n_meta']} metadata features -- build slot_meta for the TEST studies and pass it to predict() before submitting"
print(f"\n{len(models)} fold models ready | input norm: {CFG.get('norm', 'none')}")
AMP_PREF = 'bf16'

def amp_for(dev):
    if not str(dev).startswith('cuda'):
        return (torch.float32, False)
    cc = torch.cuda.get_device_capability(dev)
    if AMP_PREF == 'bf16':
        return (torch.bfloat16, True)
    if AMP_PREF == 'fp16':
        return (torch.float16, True)
    if AMP_PREF == 'fp32':
        return (torch.float32, False)
    return (torch.bfloat16 if cc >= (8, 0) else torch.float16, True)
AMP_DT, AMP_ON = amp_for(DEV)
WORKERS = max(1, min(4, os.cpu_count() or 4))
CHUNK = 48
MICRO = 8
models = [m.to(DEV).eval() for m in models]
print(f"device {DEV} | amp {str(AMP_DT).split('.')[-1]} (on={AMP_ON}) | workers {WORKERS} | chunk {CHUNK} | micro {MICRO}")

def _norm_(im):
    k = CFG.get('norm', 'none')
    if k == 'zscore':
        m = (im > 0).float()
        n = m.sum(dim=(1, 2, 3), keepdim=True).clamp(min=1.0)
        mu = (im * m).sum(dim=(1, 2, 3), keepdim=True) / n
        var = (((im - mu) * m) ** 2).sum(dim=(1, 2, 3), keepdim=True) / n
        return (im - mu) / (var.sqrt() + 1e-06) * m
    if k == 'imagenet':
        m = (im > 0).float()
        return (im - 0.485) / 0.229 * m
    return im

@torch.no_grad()
def _micro(images, masks):
    dev = DEV
    ims, slots, sidx, vms = ([], [], [], [])
    for b in range(len(masks)):
        present = np.nonzero(masks[b] > 0)[0]
        if len(present) == 0:
            continue
        blk = images[b][present]
        ims.append(torch.from_numpy(blk))
        vms.append(torch.from_numpy(blk.reshape(blk.shape[0], blk.shape[1], -1).max(2) > 0))
        slots.append(torch.from_numpy(present + 1).long())
        sidx.append(torch.full((len(present),), b, dtype=torch.long))
    out = np.full((len(models), len(masks), len(LABELS)), np.nan, np.float32)
    if not ims:
        return out
    im = _norm_(torch.cat(ims).to(dev, non_blocking=True).float().div_(255.0))
    sl = torch.cat(slots).to(dev)
    si = torch.cat(sidx).to(dev)
    vm = torch.cat(vms).to(dev)
    sm = torch.zeros(len(sl), CFG.get('n_meta', 0), device=dev)
    per = torch.zeros(len(models), len(masks), len(LABELS), device=dev, dtype=torch.float32)
    with torch.autocast('cuda' if str(dev).startswith('cuda') else 'cpu', dtype=AMP_DT, enabled=AMP_ON):
        for fold_index, model in enumerate(models):
            per[fold_index] = torch.sigmoid(model(im, sl, sm, si, len(masks), vm=vm).float())
    got = per.cpu().numpy()
    keep = np.array([(masks[b] > 0).any() for b in range(len(masks))])
    out[:, keep] = got[:, keep]
    return out

def predict(images, masks):
    out = np.full((len(models), len(masks), len(LABELS)), np.nan, np.float32)
    for a in range(0, len(masks), MICRO):
        b = min(a + MICRO, len(masks))
        out[:, a:b] = _micro(images[a:b], masks[a:b])
    return out
preds = np.full((len(models), len(studies), len(LABELS)), np.nan, np.float32)
t0, done = (time.time(), 0)
with ProcessPoolExecutor(max_workers=WORKERS) as ex:
    for c0 in range(0, len(studies), CHUNK):
        block = studies[c0:c0 + CHUNK]
        imgs = np.zeros((len(block), N_SLOT, N_SLICE, SIZE, SIZE), np.uint8)
        msks = np.zeros((len(block), N_SLOT), np.uint8)
        futs = [ex.submit(build_study, (i, s, by.get(s, []))) for i, s in enumerate(block)]
        for f in as_completed(futs):
            try:
                i, a, k = f.result()
                imgs[i], msks[i] = (a, k)
            except Exception as e:
                print(f'  study failed: {type(e).__name__}: {e}')
        preds[:, c0:c0 + len(block)] = predict(imgs, msks)
        done += len(block)
        el = time.time() - t0
        print(f'  {done:,}/{len(studies):,}  {el / 60:.1f}m  eta {el / done * (len(studies) - done) / 60:.1f}m', flush=True)
        del imgs, msks
        gc.collect()
print(f'\ninference done in {(time.time() - t0) / 60:.1f} min')
A5_W = 0.45
A5_LABELS = list(LABELS)
_a5_ok = np.isfinite(preds).all(axis=(0, 2))

_a5_rank_mean = np.zeros((len(studies), len(LABELS)), np.float64)
_v8_fold_ranks = []

for fold_index in range(preds.shape[0]):
    fold = preds[fold_index][_a5_ok]
    ordinal = fold.argsort(0).argsort(0).astype(np.float64)
    rank = ordinal / max(len(fold) - 1, 1)
    _a5_rank_mean[_a5_ok] += rank

    full_rank = np.full(
        (len(studies), len(LABELS)),
        np.nan,
        np.float64,
    )
    full_rank[_a5_ok] = rank
    _v8_fold_ranks.append(full_rank)

_a5_rank_mean /= preds.shape[0]
_a5_rank_mean[~_a5_ok] = np.nan

_v8_prob_mean = np.nanmean(preds, axis=0)
_v8_prob_rank = np.full_like(_a5_rank_mean, np.nan, dtype=np.float64)

if _a5_ok.any():
    _v8_prob_rank[_a5_ok] = pd.DataFrame(
        _v8_prob_mean[_a5_ok],
        columns=A5_LABELS,
    ).rank(
        method='average',
        pct=True,
    ).to_numpy(np.float64)

_v8_stack = np.stack(_v8_fold_ranks)
_v8_stability = np.zeros(len(LABELS), np.float64)
_v8_pairs = 0

for _i in range(len(_v8_stack)):
    for _j in range(_i + 1, len(_v8_stack)):
        for _t in range(len(LABELS)):
            _x = _v8_stack[_i, _a5_ok, _t]
            _y = _v8_stack[_j, _a5_ok, _t]
            if len(_x) > 2 and np.std(_x) > 0 and np.std(_y) > 0:
                _v8_stability[_t] += float(np.corrcoef(_x, _y)[0, 1])
        _v8_pairs += 1

_v8_stability /= max(_v8_pairs, 1)

A5_PREDS = dict(
    zip(
        sub_df['StudyInstanceUID'].astype(str),
        _a5_rank_mean.astype(np.float32),
    )
)
V8_D3_PROB_PREDS = dict(
    zip(
        sub_df['StudyInstanceUID'].astype(str),
        _v8_prob_rank.astype(np.float32),
    )
)
V8_D3_STABILITY = _v8_stability.astype(np.float64)

for _a5k, _a5v in _A5_SAVED.items():
    globals()[_a5k] = _a5v
del _A5_SAVED, _a5k, _a5v

_a5_sub = pd.read_csv(
    '/kaggle/working/submission.csv',
    dtype={'StudyInstanceUID': str},
)
assert _a5_sub.columns.tolist()[1:] == A5_LABELS, 'submission schema drift'

if A5_W > 0:
    _ids = _a5_sub['StudyInstanceUID'].astype(str).tolist()

    _a5_ours = np.stack([A5_PREDS[_u] for _u in _ids])
    _a5_prob = np.stack([V8_D3_PROB_PREDS[_u] for _u in _ids])

    _a5_base_rank = _a5_sub[A5_LABELS].rank(
        method='average',
        pct=True,
    )
    _a5_ours_rank = pd.DataFrame(
        _a5_ours,
        columns=A5_LABELS,
        index=_a5_sub.index,
    ).rank(
        method='average',
        pct=True,
    )

    V8_D2_PUBLIC_RANK = _a5_base_rank.to_numpy(np.float64)
    V8_D3_FOLD_RANK = _a5_ours_rank.to_numpy(np.float64)
    V8_D3_PROB_RANK = pd.DataFrame(
        _a5_prob,
        columns=A5_LABELS,
        index=_a5_sub.index,
    ).rank(
        method='average',
        pct=True,
    ).to_numpy(np.float64)

    def _v8_align_frame(frame):
        aligned = _a5_sub[['StudyInstanceUID']].merge(
            frame[['StudyInstanceUID'] + A5_LABELS],
            on='StudyInstanceUID',
            how='left',
            validate='one_to_one',
        )
        values = aligned[A5_LABELS].rank(
            method='average',
            pct=True,
        ).to_numpy(np.float64)
        if not np.isfinite(values).all():
            raise RuntimeError('V8 DINO alternative alignment failed')
        return values

    V8_D2_WEIGHTED_RANK = _v8_align_frame(V8_D2_WEIGHTED_FRAME)
    V8_D2_SOFT_RANK = _v8_align_frame(V8_D2_SOFT_FRAME)

    _a5_sub[A5_LABELS] = (
        (1.0 - A5_W) * _a5_base_rank
        + A5_W * _a5_ours_rank
    )

    assert np.isfinite(_a5_sub[A5_LABELS].to_numpy()).all()
    _a5_sub.to_csv('/kaggle/working/submission.csv', index=False)


### 【解説】セル3: RadImageNet + 3段のメタスタック（v10 → v11 → v12）— 「挑戦者」を立てて審査する

**何をしているか**: **医用画像で事前学習された ResNet-50（RadImageNet）**で特徴を抽出し、
その上に**メタモデルを3段階（v10, v11, v12）で重ねて**、それぞれが前段への「挑戦者(challenger)」として
採用審査を受ける、という構成です。関数44個・約11万文字で、このnotebookで最も複雑な部分です。

**なぜそうするのか**:

**① 事前学習ドメインで多様性を作る**
セル1のDINOv2は**自然画像**で自己教師あり学習、こちらのRadImageNetは**医用画像**で教師あり学習です。
同じアーキテクチャのシード違いを10本増やすより、**学んだ世界そのものが違うモデルを1本足す**ほうが、
間違え方が相関せず、アンサンブルの利得が大きくなります。

**② SHA-256 によるアセット検証（`_rad_sha256`, `_RAD_ENCODER_SHA256` など）**
重みファイルを**ハッシュで照合してから**読み込みます。データセットが更新されて中身が変わっていたら止まります。
外部データセットに依存する推論では、**気づかないまま別の重みで走る**のが最も怖い失敗です。

**③ メタスタックの段階構成**

| 段 | 中身 | 追加している情報 |
|---|---|---|
| `_v10_*` | Ridge回帰によるメタスタック | `_v10_protocol_features`（撮像プロトコル）、`_v10_demographic_features`（年齢・性別など） |
| `_v11_*` | HistGradientBoosting＋`_v11_label_graph` | **ラベル同士の関係**（例: ACL断裂と関節液貯留は同時に起きやすい） |
| `_v12_*` | pairwise モデル＋domain HGB | `_v12_stable_label_graph`（安定なラベルグラフ）、`_v12_domain_features`（DICOMヘッダ由来） |

**★ `label_graph` が効く理由**: 12ラベルは独立ではありません。前十字靭帯が切れていれば関節液が溜まりやすく、
変形性関節症は内側・外側・膝蓋大腿で相関します。**1ラベルの予測を、他ラベルの予測を手がかりに補正する**ことで、
特に単独では当てにくいラベルの順位が改善します。

**④ ★ 最重要: 「挑戦者」の採用ゲート**
新しい段は自動的には採用されません。以下を通過して初めて重みが与えられます。

- `_v10_target_auc`: **ラベルごとに** OOF AUC を計算（全体平均ではなく個別に見る）
- `_v10_bootstrap_positive_fraction(..., repeats=280)`: OOFを280回リサンプリングし、
  **改善が正だった割合**を測る。「たまたま上がった」を弾くための統計的な検定に相当します
- `_v11_fold_consistency`: **fold間で一貫して改善しているか**。1つのfoldだけ大勝ちして平均を押し上げている、
  というよくあるパターンを検出します
- `_v10_meta_stack` / `_v11_meta_challenger` / `_v12_meta_challenger`: 上を通ったものだけが
  **ラベル別の重み**を得て最終順位に混ざる（`v12_weight` は12次元ベクトル）

これは今日のPlayground枠のnotebookが「やるべきだ」と主張していたことの**実装版**です。
**「LBが上がった」ではなく「選択に使っていないデータで一貫して上がった」を採用条件にする**。
高スコアnotebookが高スコアである理由は、モデルの新しさではなく**この審査の厳しさ**にあると読めます。

**⑤ 最終合成とフェイルクローズド検証**
`_rad_validate(final, expected_ids)` で**期待するIDが全部あるか・値が有限か**を確認してから書き出し、
余計な `submission*.csv` や診断CSVを消します。作業ディレクトリに複数のCSVが残っていると
**採点対象が意図しないファイルになる**ことがあるためです。最後に `gc.collect()` と
`torch.cuda.empty_cache()` でメモリを解放しています。

**この解説付き写しから持ち帰るべきこと**:
1. **事前学習ドメインの違い**は、シード違いよりずっと強い多様性の源になる。
2. **マルチラベルなら、混ぜ方もラベルごとに独立させる**（マクロ平均AUCの構造に合わせる）。
3. **ラベル間の相関は特徴量になる**（label graph）。
4. **新しい工夫は「挑戦者」として審査する**——ブートストラップとfold一貫性で、見かけの改善を弾く。
5. 長時間ジョブでは**予算管理・ハッシュ照合・提出前検証**が精度改善と同じくらい価値がある。


In [ ]:
from __future__ import annotations
import contextlib as _rad_contextlib
import gc as _rad_gc
import hashlib as _rad_hashlib
import json as _rad_json
import os as _rad_os
import re as _rad_re
import time as _rad_time
from concurrent.futures import ThreadPoolExecutor as _RadThreadPool
from pathlib import Path as _RadPath
import numpy as _rad_np
import pandas as _rad_pd
import pydicom as _rad_pydicom
import torch as _rad_torch
import torch.nn as _rad_nn
import torch.nn.functional as _rad_F
from torchvision.models import resnet50 as _rad_resnet50
_RAD_LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
_RAD_ALPHA = 0.5
_RAD_EXCLUDE = ("Baker's", 'Fracture')
_RAD_REFERENCE_HEADS_SHA256 = '0f465649799ecfbccaac1767844639e7ced44e1bc9babde6e4bac7c5d9b89eaa'
_RAD_ENCODER_SHA256 = '08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734'
_RAD_E13_HEADS_SHA256 = 'ad9f19af73bfdf4e49263c0e45060dc3cb239e1195039b26dc8c0a3a6bcd1a8a'
_RAD_E13_MEMBER_WEIGHT = 0.5
_RAD_V48_SECOND_ALPHA = 0.15
_RAD_TOKEN_DIM, _RAD_HEAD_DIM = (2048, 512)
_RAD_E11_SLOTS = [('SAG_NOFS', 'Sagittal', None, False), ('COR_NOFS', 'Coronal', None, False), ('AX_NOFS', 'Axial', None, False), ('SAG_FS', 'Sagittal', None, True)]
_RAD_E11_CROP_MM = 130.0
_RAD_E13_SLOTS = [('SAG_FS', 'Sagittal', None, True), ('COR_FS', 'Coronal', None, True), ('AX_FS', 'Axial', None, True), ('SAG_NOFS', 'Sagittal', None, False)]
_RAD_E13_CROP_MM = 130.0
_RAD_E13_CACHE_SLICES = 8
_RAD_E13_IMG = 224
SLOTS = [('SAG_FS', 'Sagittal', None, True), ('COR_FS', 'Coronal', None, True), ('AX_FS', 'Axial', None, True)]
N_SLOT = len(SLOTS)
CACHE_SLICES = 8

def _rad_sha256(path, chunk=8 << 20):
    digest = _rad_hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(chunk), b''):
            digest.update(block)
    return digest.hexdigest()

def _rad_find_file(name, expected_sha=None, explicit_env=None):
    files = {_RAD_ENCODER_SHA256: ASSET / 'resnet-50-radimagenet-marwan/ResNet50.pt', _RAD_REFERENCE_HEADS_SHA256: ASSET / 'rsna-knee-e9-radimagenet-heads-v15/v52_radimagenet_heads.pt', _RAD_E13_HEADS_SHA256: ASSET / 'kernel-sources/rsna-knee-e13-train/rsna_rad_e11/v52_e11_heads.pt'}
    path = files.get(expected_sha)
    if path is None or not path.is_file():
        raise FileNotFoundError(name)
    if _rad_sha256(path) != expected_sha:
        raise RuntimeError(f'hash mismatch for {path}')
    return path

class _RadEncoder(_rad_nn.Module):

    def __init__(self):
        super().__init__()
        self.backbone = _rad_nn.Sequential(*list(_rad_resnet50(weights=None).children())[:-2])

    def forward(self, image):
        return self.backbone(image).mean(dim=(2, 3))

class _RadHead(_rad_nn.Module):

    def __init__(self):
        super().__init__()
        self.project = _rad_nn.Sequential(_rad_nn.LayerNorm(_RAD_TOKEN_DIM), _rad_nn.Linear(_RAD_TOKEN_DIM, _RAD_HEAD_DIM), _rad_nn.GELU())
        self.plane = _rad_nn.Parameter(_rad_torch.randn(N_SLOT, _RAD_HEAD_DIM) * 0.01)
        self.position = _rad_nn.Parameter(_rad_torch.randn(CACHE_SLICES, _RAD_HEAD_DIM) * 0.01)
        self.query = _rad_nn.Parameter(_rad_torch.randn(len(_RAD_LABELS), _RAD_HEAD_DIM) * 0.02)
        self.attn = _rad_nn.MultiheadAttention(_RAD_HEAD_DIM, 8, dropout=0.1, batch_first=True)
        self.fuse = _rad_nn.Sequential(_rad_nn.LayerNorm(_RAD_HEAD_DIM * 4), _rad_nn.Linear(_RAD_HEAD_DIM * 4, _RAD_HEAD_DIM), _rad_nn.GELU(), _rad_nn.Dropout(0.15))
        self.weight = _rad_nn.Parameter(_rad_torch.randn(len(_RAD_LABELS), _RAD_HEAD_DIM) * 0.02)
        self.bias = _rad_nn.Parameter(_rad_torch.zeros(len(_RAD_LABELS)))

    def forward(self, feature, mask):
        token = self.project(feature.float())
        token = token.view(len(token), N_SLOT, CACHE_SLICES, _RAD_HEAD_DIM)
        token = token + self.plane[None, :, None] + self.position[None, None]
        token = token.flatten(1, 2)
        key_padding = mask <= 0
        all_empty = key_padding.all(1)
        if all_empty.any():
            key_padding = key_padding.clone()
            key_padding[all_empty, 0] = False
        query = self.query.unsqueeze(0).expand(len(token), -1, -1)
        attended = query + self.attn(query, token, token, key_padding_mask=key_padding, need_weights=False)[0]
        denominator = mask.sum(1, keepdim=True).clamp_min(1).unsqueeze(-1)
        mean = (token * mask.unsqueeze(-1)).sum(1, keepdim=True) / denominator
        mean = mean.expand(-1, len(_RAD_LABELS), -1)
        fused = self.fuse(_rad_torch.cat([attended, mean, _rad_torch.abs(attended - mean), attended * mean], dim=-1))
        return (fused * self.weight.unsqueeze(0)).sum(-1) + self.bias

def _rad_load_public_heads(device, expected_sha):
    heads_path = _rad_find_file('v52_radimagenet_heads.pt', expected_sha)
    payload = _rad_torch.load(heads_path, map_location='cpu', weights_only=True)
    expected = {'version': 'v52-radimagenet-resnet50-official-1', 'targets': _RAD_LABELS, 'encoder_sha256': _RAD_ENCODER_SHA256, 'encoder_source_commit': '0ce16f7375db4236e646829d1eca61cdb4282133', 'img': 224, 'slices_per_plane': 8, 'feature': 'global_average_pool'}
    for key, value in expected.items():
        if payload.get(key) != value:
            raise RuntimeError(f'public-v15 head contract drift for {key}')
    folds = payload.get('folds')
    if not isinstance(folds, list) or len(folds) != 5:
        raise RuntimeError('public-v15 bundle requires exactly five heads')
    if sorted((int(record.get('fold', -1)) for record in folds)) != list(range(5)):
        raise RuntimeError('public-v15 fold identity drift')
    heads = []
    for record in folds:
        head = _RadHead().to(device).eval()
        head.load_state_dict(record['state_dict'], strict=True)
        heads.append(head)
    return (heads, str(heads_path))

def _rad_load_e13_heads(device):
    heads_path = _rad_find_file('v52_e11_heads.pt', _RAD_E13_HEADS_SHA256)
    payload = _rad_torch.load(heads_path, map_location='cpu', weights_only=False)
    expected = {'version': 'e11-radimagenet-resnet50-diverse-1', 'targets': _RAD_LABELS, 'encoder_sha256': _RAD_ENCODER_SHA256, 'slots': [list(slot) for slot in _RAD_E13_SLOTS], 'crop_mm': _RAD_E13_CROP_MM, 'img': _RAD_E13_IMG, 'slices_per_plane': _RAD_E13_CACHE_SLICES, 'feature': 'global_average_pool'}
    for key, value in expected.items():
        if payload.get(key) != value:
            raise RuntimeError(f'E13 head contract drift for {key}')
    folds = payload.get('folds')
    if not isinstance(folds, list) or len(folds) != 5:
        raise RuntimeError('E13 bundle requires exactly five heads')
    if sorted((int(record.get('fold', -1)) for record in folds)) != list(range(5)):
        raise RuntimeError('E13 fold identity drift')
    heads = []
    for record in folds:
        head = _RadHead().to(device).eval()
        head.load_state_dict(record['state_dict'], strict=True)
        heads.append(head)
    return (heads, str(heads_path))

@_rad_torch.inference_mode()
def _rad_encode(encoder, pixels, slot_mask, device):
    n, slots, slices, height, width = pixels.shape
    features = _rad_np.zeros((n, slots * slices, _RAD_TOKEN_DIM), _rad_np.float16)
    token_mask = _rad_np.repeat(slot_mask[:, :, None], slices, axis=2).reshape(n, -1)
    valid = _rad_np.flatnonzero(token_mask.reshape(-1) > 0)
    flat = pixels.reshape(-1, height, width)
    batch = 192 if device.type == 'cuda' and _rad_torch.cuda.device_count() > 1 else 96 if device.type == 'cuda' else 8
    for start in range(0, len(valid), batch):
        indices = valid[start:start + batch]
        image = _rad_torch.from_numpy(flat[indices]).to(device).float().div_(127.5).sub_(1.0)
        image = image.unsqueeze(1).expand(-1, 3, -1, -1).contiguous()
        amp = _rad_torch.autocast('cuda') if device.type == 'cuda' else _rad_contextlib.nullcontext()
        with amp:
            feature = encoder(image)
        values = feature.float().cpu().numpy()
        if not _rad_np.isfinite(values).all():
            raise RuntimeError('V36 non-finite RadImageNet feature')
        features.reshape(-1, _RAD_TOKEN_DIM)[indices] = values.astype(_rad_np.float16)
    return (features, token_mask.astype(_rad_np.float32))

@_rad_torch.inference_mode()
def _rad_predict_head(head, features, masks, device, batch=64):
    predictions = []
    for start in range(0, len(features), batch):
        image = _rad_torch.from_numpy(features[start:start + batch]).to(device)
        mask = _rad_torch.from_numpy(masks[start:start + batch]).to(device)
        amp = _rad_torch.autocast('cuda') if device.type == 'cuda' else _rad_contextlib.nullcontext()
        with amp:
            predictions.append(_rad_torch.sigmoid(head(image, mask)).float().cpu())
    return _rad_torch.cat(predictions).numpy()

def _rad_rank_columns(values):
    return _rad_pd.DataFrame(_rad_np.asarray(values, dtype=_rad_np.float64)).rank(method='average', pct=True).to_numpy(_rad_np.float64)

def _rad_validate(frame, expected_ids):
    if frame.columns.tolist() != ['StudyInstanceUID', *_RAD_LABELS]:
        raise RuntimeError('V36 submission schema drift')
    ids = frame['StudyInstanceUID'].astype(str).tolist()
    if ids != list(map(str, expected_ids)) or len(ids) != len(set(ids)):
        raise RuntimeError('V36 submission study identity/order drift')
    values = frame[_RAD_LABELS].to_numpy(_rad_np.float64)
    if not _rad_np.isfinite(values).all() or values.min() < 0 or values.max() > 1:
        raise RuntimeError('V36 invalid submission values')

_V8_LOCALIZED = {
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Baker's",
    "Contusion",
    "Fracture",
}


def _v8_rank_bank(predictions):
    return _rad_np.stack([
        _rad_rank_columns(p)
        for p in predictions
    ])


def _v8_target_stability(predictions):
    bank = _v8_rank_bank(predictions)
    result = _rad_np.zeros(len(_RAD_LABELS), _rad_np.float64)
    count = 0

    for a in range(len(bank)):
        for b in range(a + 1, len(bank)):
            for target in range(len(_RAD_LABELS)):
                x = bank[a, :, target]
                y = bank[b, :, target]
                if len(x) > 2 and _rad_np.std(x) > 0 and _rad_np.std(y) > 0:
                    result[target] += float(_rad_np.corrcoef(x, y)[0, 1])
            count += 1

    return result / max(count, 1)


def _v8_robust_z(values, floor=0.015):
    values = _rad_np.asarray(values, _rad_np.float64)
    center = float(_rad_np.median(values))
    mad = float(
        _rad_np.median(_rad_np.abs(values - center))
    ) * 1.4826
    scale = max(mad, float(floor))
    return (values - center) / scale


def _v8_consensus(predictions):
    probability = _rad_np.mean(
        _rad_np.stack(predictions),
        axis=0,
    )
    probability_rank = _rad_rank_columns(probability)
    fold_rank = _v8_rank_bank(predictions).mean(axis=0)
    consensus = _rad_rank_columns(
        0.90 * probability_rank
        + 0.10 * fold_rank
    )
    stability = _v8_target_stability(predictions)
    return probability_rank, consensus, stability


def _v8_extract_metric_vector(record):
    candidates = []

    def visit(value, path=""):
        if isinstance(value, dict):
            for key, child in value.items():
                visit(child, f"{path}/{key}".lower())
            return

        if isinstance(value, (list, tuple, _rad_np.ndarray)):
            try:
                array = _rad_np.asarray(value, _rad_np.float64)
            except Exception:
                return

            if (
                array.ndim == 1
                and len(array) == len(_RAD_LABELS)
                and _rad_np.isfinite(array).all()
                and any(
                    token in path
                    for token in ("auc", "metric", "score", "valid", "/val")
                )
                and not any(
                    token in path
                    for token in ("weight", "threshold", "prior", "preval", "loss")
                )
            ):
                candidates.append((path, array))

    visit(record)

    if not candidates:
        return None

    candidates.sort(
        key=lambda item: (
            0 if "auc" in item[0] else 1,
            len(item[0]),
        )
    )
    return candidates[0][1]


def _v8_bundle_metrics(path):
    try:
        payload = _rad_torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )
    except Exception:
        return None

    folds = payload.get("folds")
    if not isinstance(folds, list):
        return None

    vectors = []
    for record in folds:
        vector = _v8_extract_metric_vector(record)
        if vector is None:
            return None
        vectors.append(vector)

    if not vectors:
        return None

    return _rad_np.mean(_rad_np.stack(vectors), axis=0)


def _v8_discover_oof_auc(kind):
    roots = [
        ASSET,
        _RadPath("/kaggle/input/kernel-sources"),
    ]
    paths = []

    for root in roots:
        if not root.exists():
            continue
        try:
            paths.extend(root.rglob("*oof*.csv"))
        except Exception:
            pass

    try:
        from sklearn.metrics import roc_auc_score
    except Exception:
        return None

    train = _rad_pd.read_csv(
        ROOT / "train.csv",
        dtype={"StudyInstanceUID": str},
    )

    best = None
    best_n = -1

    for path in paths:
        text = str(path).lower()

        if kind == "e13":
            if not ("e13" in text or "e11" in text):
                continue
        else:
            if "e13" in text or "e11" in text:
                continue
            if not (
                "e9" in text
                or "radimagenet" in text
                or "public" in text
            ):
                continue

        try:
            frame = _rad_pd.read_csv(
                path,
                dtype={"StudyInstanceUID": str},
            )
        except Exception:
            continue

        if (
            "StudyInstanceUID" not in frame.columns
            or not set(_RAD_LABELS).issubset(frame.columns)
        ):
            continue

        merged = train[
            ["StudyInstanceUID"] + _RAD_LABELS
        ].merge(
            frame[["StudyInstanceUID"] + _RAD_LABELS],
            on="StudyInstanceUID",
            how="inner",
            suffixes=("_gold", "_pred"),
        )

        auc = _rad_np.full(
            len(_RAD_LABELS),
            _rad_np.nan,
            _rad_np.float64,
        )
        usable = 0

        for j, target in enumerate(_RAD_LABELS):
            y = _rad_pd.to_numeric(
                merged[f"{target}_gold"],
                errors="coerce",
            ).to_numpy()
            p = _rad_pd.to_numeric(
                merged[f"{target}_pred"],
                errors="coerce",
            ).to_numpy()

            mask = _rad_np.isfinite(y) & _rad_np.isfinite(p)

            if (
                mask.sum() >= 8
                and len(_rad_np.unique(y[mask])) == 2
            ):
                auc[j] = roc_auc_score(y[mask], p[mask])
                usable += int(mask.sum())

        if _rad_np.isfinite(auc).sum() >= 8 and usable > best_n:
            best = auc
            best_n = usable

    return best


def _v8_parent_specialist(baseline_rank):
    output = _rad_np.asarray(
        baseline_rank,
        _rad_np.float64,
    ).copy()

    d2_soft = globals().get("V8_D2_SOFT_RANK")
    d2_weighted = globals().get("V8_D2_WEIGHTED_RANK")
    d3_probability = globals().get("V8_D3_PROB_RANK")

    if d2_soft is None or d2_weighted is None or d3_probability is None:
        return output

    d3_stability = _rad_np.asarray(
        globals().get(
            "V8_D3_STABILITY",
            _rad_np.ones(len(_RAD_LABELS)),
        ),
        _rad_np.float64,
    )
    d3_z = _v8_robust_z(d3_stability, floor=0.02)

    for j, target in enumerate(_RAD_LABELS):
        soft_w = 0.055 if target in _V8_LOCALIZED else 0.015
        prob_w = float(
            _rad_np.clip(
                0.022 + 0.012 * _rad_np.tanh(d3_z[j]),
                0.008,
                0.038,
            )
        )
        weighted_w = 0.018 if target in _V8_LOCALIZED else 0.008

        total = soft_w + prob_w + weighted_w

        output[:, j] = (
            (1.0 - total) * baseline_rank[:, j]
            + soft_w * d2_soft[:, j]
            + prob_w * d3_probability[:, j]
            + weighted_w * d2_weighted[:, j]
        )

    return _rad_rank_columns(output)



# ----------------------- V10 OOF meta stack -----------------------

def _v10_find_artifact(filename):
    matches = []

    # Search attached datasets/kernel outputs while pruning the enormous
    # competition DICOM trees. This keeps startup deterministic.
    root = _RadPath("/kaggle/input")
    if root.exists():
        for current, dirs, files in os.walk(root):
            dirs[:] = [
                d for d in dirs
                if d not in (
                    "train_series",
                    "test_series",
                    ".git",
                    "__pycache__",
                )
            ]
            if filename in files:
                candidate = (
                    _RadPath(current)
                    / filename
                )
                if candidate.is_file():
                    matches.append(
                        candidate
                    )

    work = _RadPath(
        "/kaggle/working"
    )
    if work.exists():
        candidate = work / filename
        if candidate.is_file():
            matches.append(
                candidate
            )
        for child in work.iterdir():
            if not child.is_dir():
                continue
            nested = child / filename
            if nested.is_file():
                matches.append(
                    nested
                )

    if not matches:
        return None

    unique = {}
    for candidate in matches:
        try:
            key = str(
                candidate.resolve()
            )
        except Exception:
            key = str(candidate)
        unique[key] = candidate

    ordered = sorted(
        unique.values(),
        key=lambda p: (
            len(p.parts),
            str(p),
        ),
    )
    return ordered


def _v10_rank(matrix):
    return _rad_rank_columns(
        _rad_np.asarray(
            matrix,
            dtype=_rad_np.float64,
        )
    )


def _v10_protocol_features(series, ids):
    ids = [str(x) for x in ids]
    frame = series.copy()
    frame["StudyInstanceUID"] = (
        frame["StudyInstanceUID"]
        .astype(str)
    )

    result = _rad_pd.DataFrame(
        index=_rad_pd.Index(
            ids,
            name="StudyInstanceUID",
        )
    )

    grouped = frame.groupby(
        "StudyInstanceUID",
        sort=False,
    )

    result["series_total"] = (
        grouped.size()
        .reindex(result.index)
        .fillna(0)
    )

    if "Anatomical_Plane" in frame.columns:
        for plane in (
            "Sagittal",
            "Coronal",
            "Axial",
        ):
            count = (
                frame[
                    frame["Anatomical_Plane"]
                    .astype(str)
                    .eq(plane)
                ]
                .groupby(
                    "StudyInstanceUID"
                )
                .size()
            )
            result[
                f"plane_{plane.lower()}"
            ] = (
                count
                .reindex(result.index)
                .fillna(0)
            )

    for flag_name in (
        "Fat_Suppression",
        "Fluid_Sensitive",
    ):
        if flag_name not in frame.columns:
            continue

        numeric = _rad_pd.to_numeric(
            frame[flag_name],
            errors="coerce",
        ).fillna(0)

        flag_frame = frame[
            numeric > 0
        ]

        total = (
            flag_frame.groupby(
                "StudyInstanceUID"
            )
            .size()
        )
        result[
            flag_name.lower()
        ] = (
            total
            .reindex(result.index)
            .fillna(0)
        )

        if "Anatomical_Plane" in frame.columns:
            for plane in (
                "Sagittal",
                "Coronal",
                "Axial",
            ):
                count = (
                    flag_frame[
                        flag_frame[
                            "Anatomical_Plane"
                        ]
                        .astype(str)
                        .eq(plane)
                    ]
                    .groupby(
                        "StudyInstanceUID"
                    )
                    .size()
                )
                result[
                    f"{flag_name.lower()}_"
                    f"{plane.lower()}"
                ] = (
                    count
                    .reindex(result.index)
                    .fillna(0)
                )

    values = result.to_numpy(
        dtype=_rad_np.float64
    )

    if values.shape[1] == 0:
        return _rad_np.zeros(
            (len(ids), 1),
            dtype=_rad_np.float64,
        )

    return values


def _v10_demographic_features(frame, ids):
    ids = [str(x) for x in ids]
    if "StudyInstanceUID" not in frame.columns:
        return _rad_np.zeros(
            (len(ids), 3),
            dtype=_rad_np.float64,
        )

    base = frame.copy()
    base["StudyInstanceUID"] = (
        base["StudyInstanceUID"]
        .astype(str)
    )
    base = (
        _rad_pd.DataFrame(
            {"StudyInstanceUID": ids}
        )
        .merge(
            base,
            on="StudyInstanceUID",
            how="left",
            validate="one_to_one",
        )
    )

    sex = (
        base["PatientSex"]
        .astype(str)
        .str.upper()
        if "PatientSex" in base.columns
        else _rad_pd.Series(
            [""] * len(base)
        )
    )

    return _rad_np.stack(
        [
            sex.eq("M").to_numpy(
                _rad_np.float64
            ),
            sex.eq("F").to_numpy(
                _rad_np.float64
            ),
            (
                ~sex.isin(
                    ["M", "F"]
                )
            ).to_numpy(
                _rad_np.float64
            ),
        ],
        axis=1,
    )


def _v10_load_oof():
    """
    Load three genuinely out-of-fold model families:
      1. E2 DINOv2 ensemble
      2. public RadImageNet family
      3. diverse E11 RadImageNet family

    The function is intentionally strict. Any contract mismatch disables
    V10 and leaves the already valid V8 prediction untouched.
    """
    npz_candidates = (
        _v10_find_artifact(
            "oof.npz"
        )
        or []
    )
    rad_candidates = (
        _v10_find_artifact(
            "v52_oof.csv"
        )
        or []
    )
    e11_candidates = (
        _v10_find_artifact(
            "v52_e11_oof.csv"
        )
        or []
    )

    if (
        not npz_candidates
        or not rad_candidates
        or not e11_candidates
    ):
        return None

    train = _rad_pd.read_csv(
        ROOT / "train.csv",
        dtype={
            "StudyInstanceUID": str
        },
    )
    ids = (
        train["StudyInstanceUID"]
        .astype(str)
        .to_numpy()
    )

    bundle_data = None

    for path in npz_candidates:
        try:
            with _rad_np.load(
                path,
                allow_pickle=False,
            ) as bundle:
                required = {
                    "ids",
                    "pred",
                    "y_derived",
                    "gold_mask",
                    "targets",
                }
                if not required.issubset(
                    bundle.files
                ):
                    continue

                targets = (
                    bundle["targets"]
                    .astype(str)
                    .tolist()
                )
                bundle_ids = (
                    bundle["ids"]
                    .astype(str)
                )

                if targets != list(
                    _RAD_LABELS
                ):
                    continue
                if not _rad_np.array_equal(
                    bundle_ids,
                    ids,
                ):
                    continue

                bundle_data = {
                    "pred": bundle[
                        "pred"
                    ].astype(
                        _rad_np.float64
                    ),
                    "y": bundle[
                        "y_derived"
                    ].astype(
                        _rad_np.float64
                    ),
                    "gold": bundle[
                        "gold_mask"
                    ].astype(bool),
                    "path": str(path),
                }
                break
        except Exception:
            continue

    if bundle_data is None:
        return None

    def load_csv(
        candidates,
        require_fold=True,
    ):
        for path in candidates:
            try:
                frame = _rad_pd.read_csv(
                    path,
                    dtype={
                        "StudyInstanceUID": str
                    },
                )
            except Exception:
                continue

            if not {
                "StudyInstanceUID",
                *_RAD_LABELS,
            }.issubset(
                frame.columns
            ):
                continue

            aligned = (
                train[
                    ["StudyInstanceUID"]
                ]
                .merge(
                    frame,
                    on="StudyInstanceUID",
                    how="left",
                    validate="one_to_one",
                )
            )

            if aligned[
                _RAD_LABELS
            ].isna().any().any():
                continue

            if (
                require_fold
                and "fold"
                not in aligned.columns
            ):
                continue

            return (
                aligned,
                str(path),
            )

        return (
            None,
            None,
        )

    rad_frame, rad_path = load_csv(
        rad_candidates
    )
    e11_frame, e11_path = load_csv(
        e11_candidates
    )

    if (
        rad_frame is None
        or e11_frame is None
    ):
        return None

    gold = bundle_data["gold"]

    official_gold = (
        train[
            _RAD_LABELS
        ]
        .notna()
        .all(axis=1)
        .to_numpy()
    )

    if not _rad_np.array_equal(
        gold,
        official_gold,
    ):
        return None

    y = bundle_data["y"].copy()

    if (
        y.shape
        != (
            len(train),
            len(_RAD_LABELS),
        )
        or bundle_data["pred"].shape
        != (
            len(train),
            len(_RAD_LABELS),
        )
    ):
        return None

    # Replace any non-finite weak target with the neutral value. Gold rows
    # are overwritten immediately below by official labels.
    y = _rad_np.where(
        _rad_np.isfinite(y),
        y,
        0.5,
    )

    # Official labels always override report-derived targets.
    y[gold] = (
        train.loc[
            gold,
            _RAD_LABELS,
        ]
        .to_numpy(
            _rad_np.float64
        )
    )

    fold = _rad_pd.to_numeric(
        e11_frame["fold"],
        errors="coerce",
    ).to_numpy()

    if not _rad_np.isfinite(
        fold
    ).all():
        return None

    fold = fold.astype(
        _rad_np.int64
    )

    base = _v10_rank(
        bundle_data["pred"]
    )
    public = _v10_rank(
        rad_frame[
            _RAD_LABELS
        ].to_numpy(
            _rad_np.float64
        )
    )
    pass2 = _v10_rank(
        e11_frame[
            _RAD_LABELS
        ].to_numpy(
            _rad_np.float64
        )
    )

    train_series = _rad_pd.read_csv(
        ROOT / "train_series.csv",
        dtype={
            "StudyInstanceUID": str,
            "SeriesInstanceUID": str,
        },
    )

    protocol = _v10_protocol_features(
        train_series,
        ids,
    )
    demographics = (
        _v10_demographic_features(
            train,
            ids,
        )
    )

    return {
        "train": train,
        "ids": ids,
        "y": y,
        "gold": gold,
        "fold": fold,
        "base": base,
        "public": public,
        "pass2": pass2,
        "protocol": protocol,
        "demographics": demographics,
        "paths": {
            "base": bundle_data[
                "path"
            ],
            "public": rad_path,
            "pass2": e11_path,
        },
    }


def _v10_design(
    base,
    public,
    pass2,
    protocol,
    demographics,
):
    base = _rad_np.asarray(
        base,
        _rad_np.float64,
    )
    public = _rad_np.asarray(
        public,
        _rad_np.float64,
    )
    pass2 = _rad_np.asarray(
        pass2,
        _rad_np.float64,
    )

    injury = base[
        :,
        [
            _RAD_LABELS.index("ACL"),
            _RAD_LABELS.index("MCL"),
            _RAD_LABELS.index("Contusion"),
            _RAD_LABELS.index("Fracture"),
        ],
    ].mean(axis=1, keepdims=True)

    meniscus = base[
        :,
        [
            _RAD_LABELS.index(
                "Medial Meniscus"
            ),
            _RAD_LABELS.index(
                "Lateral Meniscus"
            ),
        ],
    ].mean(axis=1, keepdims=True)

    oa = base[
        :,
        [
            _RAD_LABELS.index(
                "Medial OA"
            ),
            _RAD_LABELS.index(
                "Lateral OA"
            ),
            _RAD_LABELS.index(
                "PF OA"
            ),
        ],
    ].mean(axis=1, keepdims=True)

    inflammatory = base[
        :,
        [
            _RAD_LABELS.index(
                "Effusion"
            ),
            _RAD_LABELS.index(
                "Synovitis"
            ),
            _RAD_LABELS.index(
                "Baker's"
            ),
        ],
    ].mean(axis=1, keepdims=True)

    return _rad_np.concatenate(
        [
            base,
            public,
            pass2,
            public - base,
            pass2 - base,
            injury,
            meniscus,
            oa,
            inflammatory,
            _rad_np.asarray(
                protocol,
                _rad_np.float64,
            ),
            _rad_np.asarray(
                demographics,
                _rad_np.float64,
            ),
        ],
        axis=1,
    )


def _v10_fit_ridge(
    x,
    y,
    sample_weight,
    alpha=22.0,
):
    from sklearn.linear_model import Ridge

    x = _rad_np.asarray(
        x,
        _rad_np.float64,
    )
    y = _rad_np.asarray(
        y,
        _rad_np.float64,
    )
    weight = _rad_np.asarray(
        sample_weight,
        _rad_np.float64,
    )

    mean = x.mean(
        axis=0,
        keepdims=True,
    )
    scale = x.std(
        axis=0,
        keepdims=True,
    )
    scale = _rad_np.where(
        scale > 1e-7,
        scale,
        1.0,
    )

    z = (
        x - mean
    ) / scale

    model = Ridge(
        alpha=float(alpha),
        fit_intercept=True,
    )
    model.fit(
        z,
        y,
        sample_weight=weight,
    )

    return (
        model,
        mean,
        scale,
    )


def _v10_predict_ridge(
    fitted,
    x,
):
    model, mean, scale = fitted
    x = _rad_np.asarray(
        x,
        _rad_np.float64,
    )
    z = (
        x - mean
    ) / scale
    return model.predict(z)


def _v10_target_auc(y, p):
    from sklearn.metrics import (
        roc_auc_score
    )

    y = _rad_np.asarray(y)
    p = _rad_np.asarray(p)

    mask = (
        _rad_np.isfinite(y)
        & _rad_np.isfinite(p)
    )

    if (
        mask.sum() < 4
        or len(
            _rad_np.unique(
                y[mask]
            )
        ) < 2
    ):
        return _rad_np.nan

    return float(
        roc_auc_score(
            y[mask],
            p[mask],
        )
    )


def _v10_bootstrap_positive_fraction(
    y,
    anchor,
    candidate,
    seed,
    repeats=240,
):
    y = _rad_np.asarray(y)
    anchor = _rad_np.asarray(anchor)
    candidate = _rad_np.asarray(candidate)

    positive = _rad_np.flatnonzero(
        y > 0.5
    )
    negative = _rad_np.flatnonzero(
        y <= 0.5
    )

    if (
        len(positive) < 2
        or len(negative) < 2
    ):
        return 0.0

    rng = _rad_np.random.default_rng(
        int(seed)
    )
    gains = []

    for _ in range(int(repeats)):
        pos = rng.choice(
            positive,
            size=len(positive),
            replace=True,
        )
        neg = rng.choice(
            negative,
            size=len(negative),
            replace=True,
        )
        index = _rad_np.concatenate(
            [pos, neg]
        )

        base_auc = _v10_target_auc(
            y[index],
            anchor[index],
        )
        new_auc = _v10_target_auc(
            y[index],
            candidate[index],
        )

        if (
            _rad_np.isfinite(
                base_auc
            )
            and _rad_np.isfinite(
                new_auc
            )
        ):
            gains.append(
                new_auc - base_auc
            )

    if not gains:
        return 0.0

    gains = _rad_np.asarray(
        gains,
        _rad_np.float64,
    )
    return float(
        (gains > 0).mean()
    )


def _v10_meta_stack(
    test_base,
    test_public,
    test_pass2,
    test_ids,
):
    """
    Cross-fitted, label-dependency-aware stacking.

    The stack is trained on model OOF predictions rather than in-sample
    predictions. All weak rows are available for fitting; the 58 official
    rows receive much larger weight. A target gets non-zero deployment
    weight only when its cross-fitted gold prediction improves the OOF
    anchor with bootstrap support.
    """
    data = _v10_load_oof()

    if data is None:
        return (
            None,
            _rad_np.zeros(
                len(_RAD_LABELS),
                _rad_np.float64,
            ),
        )

    train = data["train"]
    y = data["y"]
    gold = data["gold"]
    fold = data["fold"]

    x_train = _v10_design(
        data["base"],
        data["public"],
        data["pass2"],
        data["protocol"],
        data["demographics"],
    )

    test_series = _rad_pd.read_csv(
        ROOT / "test_series.csv",
        dtype={
            "StudyInstanceUID": str,
            "SeriesInstanceUID": str,
        },
    )
    test_frame = _rad_pd.read_csv(
        ROOT / "test.csv",
        dtype={
            "StudyInstanceUID": str
        },
    )

    test_protocol = (
        _v10_protocol_features(
            test_series,
            test_ids,
        )
    )
    test_demographics = (
        _v10_demographic_features(
            test_frame,
            test_ids,
        )
    )

    x_test = _v10_design(
        test_base,
        test_public,
        test_pass2,
        test_protocol,
        test_demographics,
    )

    # Approximation of the strong pre-E13 OOF route.
    oof_e10 = _v10_rank(
        0.50 * data["base"]
        + 0.50 * data["public"]
    )
    oof_anchor = _v10_rank(
        0.85 * oof_e10
        + 0.15 * data["pass2"]
    )

    meta_cross = _rad_np.full(
        y.shape,
        _rad_np.nan,
        _rad_np.float64,
    )

    # Weak report labels get lower weight when close to 0.5.
    weak_conf = (
        0.40
        + 1.60
        * _rad_np.abs(
            y - 0.5
        )
        * 2.0
    )
    weak_conf = _rad_np.clip(
        weak_conf,
        0.40,
        2.0,
    )

    unique_folds = sorted(
        int(v)
        for v in _rad_np.unique(
            fold[gold]
        )
        if int(v) >= 0
    )

    if len(unique_folds) < 3:
        return (
            None,
            _rad_np.zeros(
                len(_RAD_LABELS),
                _rad_np.float64,
            ),
        )

    for outer in unique_folds:
        train_mask = (
            fold != int(outer)
        )
        valid_mask = (
            gold
            & (fold == int(outer))
        )

        if not valid_mask.any():
            continue

        for target in range(
            len(_RAD_LABELS)
        ):
            target_y = y[:, target]
            weight = weak_conf[
                :, target
            ].copy()

            # Official labels dominate the weak report targets.
            weight[gold] = 14.0

            fitted = _v10_fit_ridge(
                x_train[train_mask],
                target_y[train_mask],
                weight[train_mask],
                alpha=24.0,
            )

            meta_cross[
                valid_mask,
                target,
            ] = _v10_predict_ridge(
                fitted,
                x_train[valid_mask],
            )

    gold_index = _rad_np.flatnonzero(
        gold
    )

    if not _rad_np.isfinite(
        meta_cross[
            gold_index
        ]
    ).all():
        return (
            None,
            _rad_np.zeros(
                len(_RAD_LABELS),
                _rad_np.float64,
            ),
        )

    meta_cross_rank = _v10_rank(
        meta_cross[
            gold_index
        ]
    )
    anchor_gold = _v10_rank(
        oof_anchor[
            gold_index
        ]
    )
    gold_y = y[
        gold_index
    ]

    deployment = _rad_np.zeros(
        len(_RAD_LABELS),
        _rad_np.float64,
    )

    grid = _rad_np.asarray(
        [
            0.00,
            0.05,
            0.10,
            0.15,
            0.20,
            0.25,
            0.30,
        ],
        _rad_np.float64,
    )

    for target in range(
        len(_RAD_LABELS)
    ):
        truth = gold_y[
            :, target
        ]

        base_auc = _v10_target_auc(
            truth,
            anchor_gold[
                :, target
            ],
        )

        if not _rad_np.isfinite(
            base_auc
        ):
            continue

        best_weight = 0.0
        best_auc = base_auc

        for weight in grid[1:]:
            candidate = (
                (1.0 - weight)
                * anchor_gold[
                    :, target
                ]
                + weight
                * meta_cross_rank[
                    :, target
                ]
            )

            auc = _v10_target_auc(
                truth,
                candidate,
            )

            # Small complexity penalty discourages large meta votes.
            penalized = (
                auc
                - 0.0025
                * float(weight)
            )

            current = (
                best_auc
                - 0.0025
                * float(best_weight)
            )

            if penalized > current:
                best_auc = auc
                best_weight = float(
                    weight
                )

        if best_weight <= 0:
            continue

        best_candidate = (
            (1.0 - best_weight)
            * anchor_gold[
                :, target
            ]
            + best_weight
            * meta_cross_rank[
                :, target
            ]
        )

        improvement = (
            best_auc - base_auc
        )

        support = (
            _v10_bootstrap_positive_fraction(
                truth,
                anchor_gold[
                    :, target
                ],
                best_candidate,
                seed=1017 + target,
            )
        )

        # Conservative gate: only deploy signal that is visible in
        # cross-fitted official labels and stable to class-stratified
        # bootstrap resampling.
        if (
            improvement >= 0.006
            and support >= 0.62
        ):
            deployment[target] = min(
                best_weight,
                0.28,
            )
        elif (
            improvement >= 0.003
            and support >= 0.58
        ):
            deployment[target] = min(
                best_weight,
                0.12,
            )

    # Expose the exact cross-fitted V10 gold ordering for the next
    # meta layer. This is validation-only state; it is never written out.
    v10_gold_final = _v10_rank(
        (
            1.0
            - deployment
        )[None, :]
        * anchor_gold
        + deployment[None, :]
        * meta_cross_rank
    )

    globals()["V11_V10_GOLD_CONTEXT"] = {
        "gold_index": gold_index.copy(),
        "gold_y": gold_y.copy(),
        "gold_fold": fold[gold_index].copy(),
        "v10_gold": v10_gold_final.copy(),
        "deployment": deployment.copy(),
    }

    if not (
        deployment > 0
    ).any():
        return (
            None,
            deployment,
        )

    meta_test = _rad_np.zeros(
        (
            len(test_ids),
            len(_RAD_LABELS),
        ),
        _rad_np.float64,
    )

    for target in range(
        len(_RAD_LABELS)
    ):
        target_y = y[:, target]
        weight = weak_conf[
            :, target
        ].copy()
        weight[gold] = 14.0

        fitted = _v10_fit_ridge(
            x_train,
            target_y,
            weight,
            alpha=24.0,
        )

        meta_test[
            :, target
        ] = _v10_predict_ridge(
            fitted,
            x_test,
        )

    return (
        _v10_rank(
            meta_test
        ),
        deployment,
    )




# ----------------------- V11 graph + nonlinear OOF stack -----------------------

def _v11_label_graph(y):
    """
    Learn a conservative positive label-correlation graph.

    The diagonal is removed so a target cannot simply copy itself.
    Only the strongest four positive neighbours per target survive.
    """
    values = _rad_np.asarray(
        y,
        dtype=_rad_np.float64,
    )

    if values.ndim != 2 or values.shape[1] != len(_RAD_LABELS):
        return _rad_np.zeros(
            (len(_RAD_LABELS), len(_RAD_LABELS)),
            dtype=_rad_np.float64,
        )

    corr = _rad_np.corrcoef(
        values,
        rowvar=False,
    )
    corr = _rad_np.nan_to_num(
        corr,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )
    corr = _rad_np.maximum(
        corr,
        0.0,
    )
    _rad_np.fill_diagonal(
        corr,
        0.0,
    )

    graph = _rad_np.zeros_like(
        corr,
        dtype=_rad_np.float64,
    )

    for target in range(
        len(_RAD_LABELS)
    ):
        row = corr[target]
        order = _rad_np.argsort(
            row
        )[::-1]
        kept = [
            int(index)
            for index in order
            if row[index] >= 0.05
        ][:4]

        if kept:
            graph[
                target,
                kept,
            ] = row[kept]

    # Symmetric shrinkage makes the graph less dependent on one noisy
    # report-derived direction.
    graph = 0.5 * (
        graph
        + graph.T
    )

    denom = graph.sum(
        axis=1,
        keepdims=True,
    )
    denom = _rad_np.where(
        denom > 1e-8,
        denom,
        1.0,
    )
    graph = graph / denom

    # Keep label propagation as a feature, not a dominant prediction.
    return 0.80 * graph


def _v11_design(
    base,
    public,
    pass2,
    protocol,
    demographics,
    graph,
):
    base = _rad_np.asarray(
        base,
        _rad_np.float64,
    )
    public = _rad_np.asarray(
        public,
        _rad_np.float64,
    )
    pass2 = _rad_np.asarray(
        pass2,
        _rad_np.float64,
    )

    core = _v10_design(
        base,
        public,
        pass2,
        protocol,
        demographics,
    )

    stack = _rad_np.stack(
        [
            base,
            public,
            pass2,
        ],
        axis=2,
    )

    consensus = stack.mean(
        axis=2
    )
    disagreement = stack.std(
        axis=2
    )
    spread = (
        stack.max(axis=2)
        - stack.min(axis=2)
    )

    # Low-order cross-family interactions help the linear learner, while
    # HGB can use them as explicit agreement/reliability signals.
    pair_product = _rad_np.concatenate(
        [
            base * public,
            base * pass2,
            public * pass2,
        ],
        axis=1,
    )

    graph = _rad_np.asarray(
        graph,
        _rad_np.float64,
    )

    graph_base = (
        base
        @ graph.T
    )
    graph_public = (
        public
        @ graph.T
    )
    graph_pass2 = (
        pass2
        @ graph.T
    )
    graph_consensus = (
        consensus
        @ graph.T
    )

    return _rad_np.concatenate(
        [
            core,
            consensus,
            disagreement,
            spread,
            pair_product,
            graph_base,
            graph_public,
            graph_pass2,
            graph_consensus,
        ],
        axis=1,
    )


def _v11_fit_hgb(
    x,
    y,
    sample_weight,
    seed,
):
    from sklearn.ensemble import (
        HistGradientBoostingRegressor
    )

    model = HistGradientBoostingRegressor(
        loss="squared_error",
        learning_rate=0.035,
        max_iter=140,
        max_leaf_nodes=7,
        max_depth=3,
        min_samples_leaf=42,
        l2_regularization=9.0,
        early_stopping=False,
        random_state=int(seed),
    )
    model.fit(
        _rad_np.asarray(
            x,
            _rad_np.float64,
        ),
        _rad_np.asarray(
            y,
            _rad_np.float64,
        ),
        sample_weight=_rad_np.asarray(
            sample_weight,
            _rad_np.float64,
        ),
    )
    return model


def _v11_fold_consistency(
    truth,
    baseline,
    candidate,
    folds,
):
    gains = []

    for fold_id in sorted(
        int(value)
        for value in _rad_np.unique(
            folds
        )
    ):
        mask = (
            folds == int(fold_id)
        )

        if mask.sum() < 4:
            continue

        base_auc = _v10_target_auc(
            truth[mask],
            baseline[mask],
        )
        candidate_auc = (
            _v10_target_auc(
                truth[mask],
                candidate[mask],
            )
        )

        if (
            _rad_np.isfinite(
                base_auc
            )
            and _rad_np.isfinite(
                candidate_auc
            )
        ):
            gains.append(
                candidate_auc
                - base_auc
            )

    if not gains:
        return (
            0,
            0.0,
            0.0,
        )

    gains = _rad_np.asarray(
        gains,
        _rad_np.float64,
    )

    return (
        int(len(gains)),
        float(
            (gains > 0).mean()
        ),
        float(
            _rad_np.median(gains)
        ),
    )


def _v11_meta_challenger(
    test_base,
    test_public,
    test_pass2,
    test_ids,
):
    """
    Nonlinear challenger above V10.

    It combines:
      - the original linear Ridge stack;
      - a small HGB regressor that can learn conditional family reliability;
      - label-correlation graph features;
      - explicit disagreement / spread / pairwise family interactions.

    Deployment is target-specific and accepted only when the candidate
    beats the already cross-fitted V10 gold prediction.
    """
    context = globals().get(
        "V11_V10_GOLD_CONTEXT"
    )
    data = _v10_load_oof()

    if (
        context is None
        or data is None
    ):
        return (
            None,
            _rad_np.zeros(
                len(_RAD_LABELS),
                _rad_np.float64,
            ),
            ["none"] * len(_RAD_LABELS),
        )

    y = data["y"]
    gold = data["gold"]
    fold = data["fold"]

    gold_index = _rad_np.asarray(
        context["gold_index"],
        dtype=_rad_np.int64,
    )
    gold_y = _rad_np.asarray(
        context["gold_y"],
        dtype=_rad_np.float64,
    )
    gold_fold = _rad_np.asarray(
        context["gold_fold"],
        dtype=_rad_np.int64,
    )
    v10_gold = _rad_np.asarray(
        context["v10_gold"],
        dtype=_rad_np.float64,
    )

    expected_gold = _rad_np.flatnonzero(
        gold
    )
    if not _rad_np.array_equal(
        gold_index,
        expected_gold,
    ):
        return (
            None,
            _rad_np.zeros(
                len(_RAD_LABELS),
                _rad_np.float64,
            ),
            ["none"] * len(_RAD_LABELS),
        )

    train_series = _rad_pd.read_csv(
        ROOT / "train_series.csv",
        dtype={
            "StudyInstanceUID": str,
            "SeriesInstanceUID": str,
        },
    )
    test_series = _rad_pd.read_csv(
        ROOT / "test_series.csv",
        dtype={
            "StudyInstanceUID": str,
            "SeriesInstanceUID": str,
        },
    )
    test_frame = _rad_pd.read_csv(
        ROOT / "test.csv",
        dtype={
            "StudyInstanceUID": str
        },
    )

    train_protocol = (
        _v10_protocol_features(
            train_series,
            data["ids"],
        )
    )
    test_protocol = (
        _v10_protocol_features(
            test_series,
            test_ids,
        )
    )
    train_demographics = (
        _v10_demographic_features(
            data["train"],
            data["ids"],
        )
    )
    test_demographics = (
        _v10_demographic_features(
            test_frame,
            test_ids,
        )
    )

    weak_conf = (
        0.35
        + 1.65
        * _rad_np.abs(
            y - 0.5
        )
        * 2.0
    )
    weak_conf = _rad_np.clip(
        weak_conf,
        0.35,
        2.0,
    )

    unique_folds = sorted(
        int(value)
        for value in _rad_np.unique(
            fold[gold]
        )
        if int(value) >= 0
    )

    if len(unique_folds) < 3:
        return (
            None,
            _rad_np.zeros(
                len(_RAD_LABELS),
                _rad_np.float64,
            ),
            ["none"] * len(_RAD_LABELS),
        )

    ridge_cross = _rad_np.full(
        y.shape,
        _rad_np.nan,
        _rad_np.float64,
    )
    hgb_cross = _rad_np.full(
        y.shape,
        _rad_np.nan,
        _rad_np.float64,
    )

    for outer in unique_folds:
        train_mask = (
            fold != int(outer)
        )
        valid_mask = (
            gold
            & (fold == int(outer))
        )

        if not valid_mask.any():
            continue

        # Strictly fold-safe label graph.
        graph = _v11_label_graph(
            y[train_mask]
        )

        x_fold = _v11_design(
            data["base"],
            data["public"],
            data["pass2"],
            train_protocol,
            train_demographics,
            graph,
        )

        for target in range(
            len(_RAD_LABELS)
        ):
            target_y = y[
                :, target
            ]
            weight = weak_conf[
                :, target
            ].copy()

            # V11 trusts official labels slightly more than V10, while
            # regularization and cross-fit gates prevent memorization.
            weight[gold] = 18.0

            ridge = _v10_fit_ridge(
                x_fold[train_mask],
                target_y[train_mask],
                weight[train_mask],
                alpha=20.0,
            )
            ridge_cross[
                valid_mask,
                target,
            ] = _v10_predict_ridge(
                ridge,
                x_fold[valid_mask],
            )

            try:
                hgb = _v11_fit_hgb(
                    x_fold[train_mask],
                    target_y[train_mask],
                    weight[train_mask],
                    seed=1147
                    + 31 * int(outer)
                    + target,
                )
                hgb_cross[
                    valid_mask,
                    target,
                ] = hgb.predict(
                    x_fold[
                        valid_mask
                    ]
                )
            except Exception:
                # Ridge remains a complete fallback candidate.
                hgb_cross[
                    valid_mask,
                    target,
                ] = ridge_cross[
                    valid_mask,
                    target,
                ]

    if not (
        _rad_np.isfinite(
            ridge_cross[
                gold_index
            ]
        ).all()
        and _rad_np.isfinite(
            hgb_cross[
                gold_index
            ]
        ).all()
    ):
        return (
            None,
            _rad_np.zeros(
                len(_RAD_LABELS),
                _rad_np.float64,
            ),
            ["none"] * len(_RAD_LABELS),
        )

    ridge_gold = _v10_rank(
        ridge_cross[
            gold_index
        ]
    )
    hgb_gold = _v10_rank(
        hgb_cross[
            gold_index
        ]
    )
    blend_gold = _v10_rank(
        0.68 * ridge_gold
        + 0.32 * hgb_gold
    )

    candidate_gold = {
        "ridge": ridge_gold,
        "hgb": hgb_gold,
        "blend": blend_gold,
    }

    deployment = _rad_np.zeros(
        len(_RAD_LABELS),
        _rad_np.float64,
    )
    chosen_model = [
        "none"
        for _ in _RAD_LABELS
    ]

    grid = _rad_np.asarray(
        [
            0.05,
            0.10,
            0.15,
            0.20,
            0.25,
            0.30,
            0.35,
        ],
        _rad_np.float64,
    )

    for target in range(
        len(_RAD_LABELS)
    ):
        truth = gold_y[
            :, target
        ]
        base = v10_gold[
            :, target
        ]

        base_auc = _v10_target_auc(
            truth,
            base,
        )
        if not _rad_np.isfinite(
            base_auc
        ):
            continue

        best = None

        for model_name, matrix in (
            candidate_gold.items()
        ):
            for mix in grid:
                candidate = (
                    (1.0 - mix)
                    * base
                    + mix
                    * matrix[
                        :, target
                    ]
                )

                auc = _v10_target_auc(
                    truth,
                    candidate,
                )
                if not _rad_np.isfinite(
                    auc
                ):
                    continue

                support = (
                    _v10_bootstrap_positive_fraction(
                        truth,
                        base,
                        candidate,
                        seed=2111
                        + 97 * target
                        + int(
                            100 * mix
                        ),
                        repeats=280,
                    )
                )

                (
                    valid_folds,
                    positive_fraction,
                    median_fold_gain,
                ) = _v11_fold_consistency(
                    truth,
                    base,
                    candidate,
                    gold_fold,
                )

                # Penalize nonlinear complexity and large deployment votes.
                complexity = (
                    0.0008
                    if model_name == "hgb"
                    else (
                        0.0004
                        if model_name == "blend"
                        else 0.0
                    )
                )
                objective = (
                    auc
                    - 0.0022
                    * float(mix)
                    - complexity
                )

                record = {
                    "model": model_name,
                    "mix": float(mix),
                    "auc": float(auc),
                    "gain": float(
                        auc - base_auc
                    ),
                    "support": float(
                        support
                    ),
                    "valid_folds": int(
                        valid_folds
                    ),
                    "fold_positive": float(
                        positive_fraction
                    ),
                    "fold_median": float(
                        median_fold_gain
                    ),
                    "objective": float(
                        objective
                    ),
                }

                if (
                    best is None
                    or record["objective"]
                    > best["objective"]
                ):
                    best = record

        if best is None:
            continue

        # Strong gate against 58-study overfitting.
        fold_ok = (
            best["valid_folds"] < 2
            or best["fold_positive"]
            >= 0.60
        )

        strong = (
            best["gain"] >= 0.005
            and best["support"] >= 0.62
            and fold_ok
        )
        moderate = (
            best["gain"] >= 0.0025
            and best["support"] >= 0.59
            and fold_ok
        )

        if strong:
            deployment[target] = min(
                best["mix"],
                0.30,
            )
            chosen_model[target] = (
                best["model"]
            )
        elif moderate:
            deployment[target] = min(
                best["mix"],
                0.12,
            )
            chosen_model[target] = (
                best["model"]
            )

    chosen_gold = _rad_np.zeros_like(
        v10_gold,
        dtype=_rad_np.float64,
    )
    for target in range(
        len(_RAD_LABELS)
    ):
        model_name = chosen_model[target]
        if model_name == "none":
            chosen_gold[:, target] = v10_gold[:, target]
        else:
            chosen_gold[:, target] = candidate_gold[
                model_name
            ][:, target]

    v11_gold = _v10_rank(
        (
            1.0
            - deployment
        )[None, :]
        * v10_gold
        + deployment[None, :]
        * chosen_gold
    )

    globals()["V12_V11_GOLD_CONTEXT"] = {
        "gold_index": gold_index.copy(),
        "gold_y": gold_y.copy(),
        "gold_fold": gold_fold.copy(),
        "v11_gold": v11_gold.copy(),
        "deployment": deployment.copy(),
        "model": list(chosen_model),
    }

    if not (
        deployment > 0
    ).any():
        return (
            None,
            deployment,
            chosen_model,
        )

    full_graph = _v11_label_graph(
        y
    )
    x_train = _v11_design(
        data["base"],
        data["public"],
        data["pass2"],
        train_protocol,
        train_demographics,
        full_graph,
    )
    x_test = _v11_design(
        test_base,
        test_public,
        test_pass2,
        test_protocol,
        test_demographics,
        full_graph,
    )

    challenger = _rad_np.zeros(
        (
            len(test_ids),
            len(_RAD_LABELS),
        ),
        _rad_np.float64,
    )

    for target in range(
        len(_RAD_LABELS)
    ):
        model_name = chosen_model[
            target
        ]

        if model_name == "none":
            # Value is irrelevant because deployment is zero.
            challenger[
                :, target
            ] = test_base[
                :, target
            ]
            continue

        target_y = y[
            :, target
        ]
        weight = weak_conf[
            :, target
        ].copy()
        weight[gold] = 18.0

        ridge = _v10_fit_ridge(
            x_train,
            target_y,
            weight,
            alpha=20.0,
        )
        ridge_test = _v10_predict_ridge(
            ridge,
            x_test,
        )

        if model_name == "ridge":
            challenger[
                :, target
            ] = ridge_test
            continue

        try:
            hgb = _v11_fit_hgb(
                x_train,
                target_y,
                weight,
                seed=4171 + target,
            )
            hgb_test = hgb.predict(
                x_test
            )
        except Exception:
            hgb_test = ridge_test

        if model_name == "hgb":
            challenger[
                :, target
            ] = hgb_test
        else:
            # Same fixed mixture used for cross-fitted validation.
            challenger[
                :, target
            ] = (
                0.68 * ridge_test
                + 0.32 * hgb_test
            )

    return (
        _v10_rank(
            challenger
        ),
        deployment,
        chosen_model,
    )




# ----------------------- V12 domain + AUC rank stack -----------------------

def _v12_header_table(split):
    key = f"_V12_HEADERS_{split}"
    cached = globals().get(key)
    if cached is not None:
        return cached

    table = annotate(
        walk(split)
    )
    globals()[key] = table
    return table


def _v12_first_numeric(series):
    values = _rad_pd.to_numeric(
        series,
        errors="coerce",
    ).to_numpy(
        _rad_np.float64
    )
    return values


def _v12_domain_features(headers, ids):
    """
    Study-level scanner/protocol descriptors.

    Only broad acquisition information is used:
      - scanner manufacturer family / model hash
      - field strength
      - TR / TE / slice thickness / spacing / flip angle
      - pixel spacing, matrix/FOV, slice-count statistics
      - broad sequence-text flags

    The features are label-free and deterministic.
    """
    import hashlib as _v12_hashlib

    ids = [str(value) for value in ids]
    index = _rad_pd.Index(
        ids,
        name="StudyInstanceUID",
    )

    frame = headers.copy()
    frame["StudyInstanceUID"] = (
        frame["StudyInstanceUID"]
        .astype(str)
    )

    if "weight" not in frame.columns:
        frame = annotate(frame)

    # Parse pixel spacing into two numeric axes.
    spacing = (
        frame.get(
            "PixelSpacing",
            _rad_pd.Series(
                [""] * len(frame)
            ),
        )
        .fillna("")
        .astype(str)
        .str.split("|")
    )

    frame["_px0"] = _rad_pd.to_numeric(
        spacing.str[0],
        errors="coerce",
    )
    frame["_px1"] = _rad_pd.to_numeric(
        spacing.str[1],
        errors="coerce",
    )

    numeric_columns = [
        "n_slices",
        "RepetitionTime",
        "EchoTime",
        "MagneticFieldStrength",
        "SliceThickness",
        "SpacingBetweenSlices",
        "FlipAngle",
        "EchoTrainLength",
        "PercentSampling",
        "PercentPhaseFieldOfView",
        "Rows",
        "Columns",
        "_px0",
        "_px1",
    ]

    feature_blocks = []

    grouped = frame.groupby(
        "StudyInstanceUID",
        sort=False,
    )

    for column in numeric_columns:
        if column not in frame.columns:
            med = _rad_np.zeros(
                len(index),
                _rad_np.float64,
            )
            spread = med.copy()
            missing = _rad_np.ones(
                len(index),
                _rad_np.float64,
            )
        else:
            values = _rad_pd.to_numeric(
                frame[column],
                errors="coerce",
            )
            temp = frame[
                ["StudyInstanceUID"]
            ].copy()
            temp["_value"] = values

            group = temp.groupby(
                "StudyInstanceUID"
            )["_value"]

            med = (
                group.median()
                .reindex(index)
                .to_numpy(
                    _rad_np.float64
                )
            )

            q75 = (
                group.quantile(0.75)
                .reindex(index)
                .to_numpy(
                    _rad_np.float64
                )
            )
            q25 = (
                group.quantile(0.25)
                .reindex(index)
                .to_numpy(
                    _rad_np.float64
                )
            )
            spread = q75 - q25

            missing = (
                group.apply(
                    lambda s: float(
                        s.isna().mean()
                    )
                )
                .reindex(index)
                .fillna(1.0)
                .to_numpy(
                    _rad_np.float64
                )
            )

            med = _rad_np.nan_to_num(
                med,
                nan=0.0,
                posinf=0.0,
                neginf=0.0,
            )
            spread = _rad_np.nan_to_num(
                spread,
                nan=0.0,
                posinf=0.0,
                neginf=0.0,
            )

        # log1p keeps long-tailed protocol values numerically tame.
        feature_blocks.extend(
            [
                _rad_np.sign(med)
                * _rad_np.log1p(
                    _rad_np.abs(med)
                ),
                _rad_np.log1p(
                    _rad_np.maximum(
                        spread,
                        0.0,
                    )
                ),
                missing,
            ]
        )

    # Derived in-plane FOV statistics from the series median.
    def study_median(column):
        if column not in frame.columns:
            return _rad_np.zeros(
                len(index),
                _rad_np.float64,
            )
        return _rad_np.nan_to_num(
            _rad_pd.to_numeric(
                frame[column],
                errors="coerce",
            )
            .groupby(
                frame[
                    "StudyInstanceUID"
                ]
            )
            .median()
            .reindex(index)
            .to_numpy(
                _rad_np.float64
            ),
            nan=0.0,
        )

    rows = study_median("Rows")
    cols = study_median("Columns")
    px0 = study_median("_px0")
    px1 = study_median("_px1")

    feature_blocks.extend(
        [
            _rad_np.log1p(
                _rad_np.maximum(
                    rows * px0,
                    0.0,
                )
            ),
            _rad_np.log1p(
                _rad_np.maximum(
                    cols * px1,
                    0.0,
                )
            ),
            _rad_np.log1p(
                _rad_np.maximum(
                    px0 * px1,
                    0.0,
                )
            ),
        ]
    )

    # Broad vendor family.
    manufacturer = (
        frame[
            [
                "StudyInstanceUID",
                "Manufacturer",
            ]
        ]
        if "Manufacturer" in frame.columns
        else _rad_pd.DataFrame(
            {
                "StudyInstanceUID":
                    frame["StudyInstanceUID"],
                "Manufacturer": "",
            }
        )
    )

    def mode_text(series):
        values = [
            str(value).strip().upper()
            for value in series
            if str(value).strip()
            not in ("", "NAN", "NONE")
        ]
        if not values:
            return ""
        return max(
            set(values),
            key=values.count,
        )

    vendor_text = (
        manufacturer.groupby(
            "StudyInstanceUID"
        )["Manufacturer"]
        .apply(mode_text)
        .reindex(index)
        .fillna("")
    )

    vendor_matrix = _rad_np.zeros(
        (len(index), 6),
        _rad_np.float64,
    )

    for row_index, text in enumerate(
        vendor_text.astype(str)
    ):
        upper = text.upper()

        if (
            "SIEMENS" in upper
            or "HEALTHINEERS" in upper
        ):
            bucket = 0
        elif (
            "GE " in upper
            or upper.startswith("GE")
            or "GENERAL ELECTRIC"
            in upper
        ):
            bucket = 1
        elif "PHILIPS" in upper:
            bucket = 2
        elif (
            "CANON" in upper
            or "TOSHIBA" in upper
        ):
            bucket = 3
        elif upper:
            bucket = 4
        else:
            bucket = 5

        vendor_matrix[
            row_index,
            bucket,
        ] = 1.0

    feature_blocks.extend(
        [
            vendor_matrix[:, j]
            for j in range(
                vendor_matrix.shape[1]
            )
        ]
    )

    # Stable low-cardinality scanner-model hash. This avoids hard coding
    # model names and maps unseen models to deterministic buckets.
    model_text = (
        frame.groupby(
            "StudyInstanceUID"
        )["ManufacturerModelName"]
        .apply(mode_text)
        .reindex(index)
        .fillna("")
        if "ManufacturerModelName"
        in frame.columns
        else _rad_pd.Series(
            [""] * len(index),
            index=index,
        )
    )

    model_hash = _rad_np.zeros(
        (len(index), 8),
        _rad_np.float64,
    )

    for row_index, text in enumerate(
        model_text.astype(str)
    ):
        if not text:
            continue

        digest = _v12_hashlib.md5(
            text.encode(
                "utf-8",
                errors="ignore",
            )
        ).hexdigest()
        bucket = int(
            digest[:8],
            16,
        ) % model_hash.shape[1]
        model_hash[
            row_index,
            bucket,
        ] = 1.0

    feature_blocks.extend(
        [
            model_hash[:, j]
            for j in range(
                model_hash.shape[1]
            )
        ]
    )

    # Field-strength bins are clinically meaningful and robust to exact
    # scanner model naming.
    field = study_median(
        "MagneticFieldStrength"
    )
    field_matrix = _rad_np.zeros(
        (len(index), 4),
        _rad_np.float64,
    )

    for row_index, value in enumerate(
        field
    ):
        if value <= 0:
            bucket = 3
        elif abs(value - 1.5) <= 0.35:
            bucket = 0
        elif abs(value - 3.0) <= 0.45:
            bucket = 1
        else:
            bucket = 2
        field_matrix[
            row_index,
            bucket,
        ] = 1.0

    feature_blocks.extend(
        [
            field_matrix[:, j]
            for j in range(
                field_matrix.shape[1]
            )
        ]
    )

    # Broad sequence-family text indicators.
    description = (
        frame.get(
            "SeriesDescription",
            _rad_pd.Series(
                [""] * len(frame)
            ),
        )
        .fillna("")
        .astype(str)
        + " "
        + frame.get(
            "SequenceName",
            _rad_pd.Series(
                [""] * len(frame)
            ),
        )
        .fillna("")
        .astype(str)
    ).str.lower()

    text_flags = {
        "stir": r"\bstir\b",
        "dixon": r"dixon",
        "three_d": r"\b3d\b",
        "cube": r"\bcube\b",
        "vista": r"\bvista\b",
        "space": r"\bspace\b",
        "tse": r"\btse\b",
        "fse": r"\bfse\b",
        "gre": r"\bgre\b|gradient",
    }

    for pattern in text_flags.values():
        matched = description.str.contains(
            pattern,
            regex=True,
            na=False,
        )
        count = (
            matched.astype(
                _rad_np.float64
            )
            .groupby(
                frame[
                    "StudyInstanceUID"
                ]
            )
            .sum()
            .reindex(index)
            .fillna(0.0)
            .to_numpy(
                _rad_np.float64
            )
        )
        feature_blocks.append(
            _rad_np.log1p(count)
        )

    matrix = _rad_np.stack(
        feature_blocks,
        axis=1,
    )

    return _rad_np.nan_to_num(
        matrix,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )


def _v12_stable_label_graph(
    y,
    base,
    folds,
    train_mask,
):
    """
    Retain label edges that are positive in both supervision space and
    image-OOF space across multiple folds. This is stricter than V11's
    single global correlation matrix.
    """
    y = _rad_np.asarray(
        y,
        _rad_np.float64,
    )
    base = _rad_np.asarray(
        base,
        _rad_np.float64,
    )
    folds = _rad_np.asarray(
        folds,
    )
    train_mask = _rad_np.asarray(
        train_mask,
        dtype=bool,
    )

    y_corrs = []
    b_corrs = []

    for fold_id in sorted(
        int(value)
        for value in _rad_np.unique(
            folds[train_mask]
        )
    ):
        mask = (
            train_mask
            & (folds == int(fold_id))
        )

        if mask.sum() < 40:
            continue

        yc = _rad_np.corrcoef(
            y[mask],
            rowvar=False,
        )
        bc = _rad_np.corrcoef(
            base[mask],
            rowvar=False,
        )

        y_corrs.append(
            _rad_np.nan_to_num(
                yc,
                nan=0.0,
            )
        )
        b_corrs.append(
            _rad_np.nan_to_num(
                bc,
                nan=0.0,
            )
        )

    if len(y_corrs) < 2:
        return _v11_label_graph(
            y[train_mask]
        )

    y_stack = _rad_np.stack(
        y_corrs
    )
    b_stack = _rad_np.stack(
        b_corrs
    )

    y_med = _rad_np.median(
        y_stack,
        axis=0,
    )
    b_med = _rad_np.median(
        b_stack,
        axis=0,
    )

    y_positive = (
        y_stack > 0
    ).mean(axis=0)
    b_positive = (
        b_stack > 0
    ).mean(axis=0)

    strength = _rad_np.sqrt(
        _rad_np.maximum(
            y_med,
            0.0,
        )
        * _rad_np.maximum(
            b_med,
            0.0,
        )
    )

    strength[
        (y_positive < 0.60)
        | (b_positive < 0.60)
    ] = 0.0

    _rad_np.fill_diagonal(
        strength,
        0.0,
    )

    graph = _rad_np.zeros_like(
        strength
    )

    for target in range(
        len(_RAD_LABELS)
    ):
        row = strength[target]
        order = _rad_np.argsort(
            row
        )[::-1]
        keep = [
            int(index)
            for index in order
            if row[index] >= 0.035
        ][:3]

        if keep:
            graph[
                target,
                keep,
            ] = row[keep]

    graph = 0.5 * (
        graph
        + graph.T
    )

    denom = graph.sum(
        axis=1,
        keepdims=True,
    )
    denom = _rad_np.where(
        denom > 1e-8,
        denom,
        1.0,
    )

    return 0.72 * (
        graph / denom
    )


def _v12_design(
    base,
    public,
    pass2,
    protocol,
    demographics,
    domain,
    graph,
):
    core = _v11_design(
        base,
        public,
        pass2,
        protocol,
        demographics,
        graph,
    )

    base = _rad_np.asarray(
        base,
        _rad_np.float64,
    )
    public = _rad_np.asarray(
        public,
        _rad_np.float64,
    )
    pass2 = _rad_np.asarray(
        pass2,
        _rad_np.float64,
    )
    domain = _rad_np.asarray(
        domain,
        _rad_np.float64,
    )

    # Study-level family disagreement summarizes when scanner/protocol
    # characteristics may change which model family should be trusted.
    d_bp = _rad_np.mean(
        _rad_np.abs(
            base - public
        ),
        axis=1,
        keepdims=True,
    )
    d_bs = _rad_np.mean(
        _rad_np.abs(
            base - pass2
        ),
        axis=1,
        keepdims=True,
    )
    d_ps = _rad_np.mean(
        _rad_np.abs(
            public - pass2
        ),
        axis=1,
        keepdims=True,
    )

    disagreement = _rad_np.concatenate(
        [
            d_bp,
            d_bs,
            d_ps,
            _rad_np.maximum(
                _rad_np.maximum(
                    d_bp,
                    d_bs,
                ),
                d_ps,
            ),
        ],
        axis=1,
    )

    # Explicit domain × disagreement interactions are compact and let even
    # the linear pairwise ranker condition its family weighting on scanner
    # characteristics.
    interactions = _rad_np.concatenate(
        [
            domain
            * disagreement[
                :, [index]
            ]
            for index in range(
                disagreement.shape[1]
            )
        ],
        axis=1,
    )

    return _rad_np.concatenate(
        [
            core,
            domain,
            disagreement,
            interactions,
        ],
        axis=1,
    )


def _v12_fit_pairwise(
    x,
    y,
    train_mask,
    gold,
    seed,
):
    from sklearn.linear_model import (
        LogisticRegression
    )

    x = _rad_np.asarray(
        x,
        _rad_np.float64,
    )
    y = _rad_np.asarray(
        y,
        _rad_np.float64,
    )
    train_mask = _rad_np.asarray(
        train_mask,
        dtype=bool,
    )
    gold = _rad_np.asarray(
        gold,
        dtype=bool,
    )

    valid = (
        train_mask
        & _rad_np.isfinite(y)
    )

    positive = _rad_np.flatnonzero(
        valid
        & (
            (
                gold
                & (y > 0.5)
            )
            | (
                (~gold)
                & (y >= 0.82)
            )
        )
    )
    negative = _rad_np.flatnonzero(
        valid
        & (
            (
                gold
                & (y <= 0.5)
            )
            | (
                (~gold)
                & (y <= 0.18)
            )
        )
    )

    if (
        len(positive) < 5
        or len(negative) < 5
    ):
        return None

    rng = _rad_np.random.default_rng(
        int(seed)
    )

    n_pairs = int(
        min(
            9000,
            max(
                3000,
                35
                * (
                    len(positive)
                    + len(negative)
                ),
            ),
        )
    )

    pos_index = rng.choice(
        positive,
        size=n_pairs,
        replace=True,
    )
    neg_index = rng.choice(
        negative,
        size=n_pairs,
        replace=True,
    )

    scale = x[
        valid
    ].std(
        axis=0,
        keepdims=True,
    )
    scale = _rad_np.where(
        scale > 1e-7,
        scale,
        1.0,
    )

    difference = (
        x[pos_index]
        - x[neg_index]
    ) / scale

    train_x = _rad_np.concatenate(
        [
            difference,
            -difference,
        ],
        axis=0,
    )
    train_y = _rad_np.concatenate(
        [
            _rad_np.ones(
                n_pairs,
                _rad_np.int64,
            ),
            _rad_np.zeros(
                n_pairs,
                _rad_np.int64,
            ),
        ]
    )

    positive_conf = _rad_np.where(
        gold[pos_index],
        3.0,
        _rad_np.clip(
            (
                y[pos_index]
                - 0.5
            )
            * 2.0,
            0.30,
            1.0,
        ),
    )
    negative_conf = _rad_np.where(
        gold[neg_index],
        3.0,
        _rad_np.clip(
            (
                0.5
                - y[neg_index]
            )
            * 2.0,
            0.30,
            1.0,
        ),
    )

    pair_weight = _rad_np.sqrt(
        positive_conf
        * negative_conf
    )
    train_weight = _rad_np.concatenate(
        [
            pair_weight,
            pair_weight,
        ]
    )

    model = LogisticRegression(
        C=0.075,
        penalty="l2",
        solver="liblinear",
        fit_intercept=False,
        max_iter=220,
        random_state=int(seed),
    )
    model.fit(
        train_x,
        train_y,
        sample_weight=train_weight,
    )

    return (
        model,
        scale,
    )


def _v12_predict_pairwise(
    fitted,
    x,
):
    if fitted is None:
        return None

    model, scale = fitted

    return model.decision_function(
        _rad_np.asarray(
            x,
            _rad_np.float64,
        )
        / scale
    )


def _v12_fit_domain_hgb(
    x,
    y,
    sample_weight,
    seed,
):
    from sklearn.ensemble import (
        HistGradientBoostingRegressor
    )

    model = HistGradientBoostingRegressor(
        loss="squared_error",
        learning_rate=0.030,
        max_iter=115,
        max_leaf_nodes=7,
        max_depth=3,
        min_samples_leaf=48,
        l2_regularization=12.0,
        early_stopping=False,
        random_state=int(seed),
    )
    model.fit(
        _rad_np.asarray(
            x,
            _rad_np.float64,
        ),
        _rad_np.asarray(
            y,
            _rad_np.float64,
        ),
        sample_weight=_rad_np.asarray(
            sample_weight,
            _rad_np.float64,
        ),
    )
    return model


def _v12_meta_challenger(
    test_base,
    test_public,
    test_pass2,
    test_ids,
):
    context = globals().get(
        "V12_V11_GOLD_CONTEXT"
    )
    data = _v10_load_oof()

    if (
        context is None
        or data is None
    ):
        return (
            None,
            _rad_np.zeros(
                len(_RAD_LABELS),
                _rad_np.float64,
            ),
            ["none"] * len(_RAD_LABELS),
        )

    y = data["y"]
    gold = data["gold"]
    fold = data["fold"]

    gold_index = _rad_np.asarray(
        context["gold_index"],
        dtype=_rad_np.int64,
    )
    gold_y = _rad_np.asarray(
        context["gold_y"],
        _rad_np.float64,
    )
    gold_fold = _rad_np.asarray(
        context["gold_fold"],
        dtype=_rad_np.int64,
    )
    v11_gold = _rad_np.asarray(
        context["v11_gold"],
        _rad_np.float64,
    )

    if not _rad_np.array_equal(
        gold_index,
        _rad_np.flatnonzero(gold),
    ):
        return (
            None,
            _rad_np.zeros(
                len(_RAD_LABELS),
                _rad_np.float64,
            ),
            ["none"] * len(_RAD_LABELS),
        )

    train_headers = _v12_header_table(
        "train_series"
    )
    test_headers = _v12_header_table(
        "test_series"
    )

    train_domain = _v12_domain_features(
        train_headers,
        data["ids"],
    )
    test_domain = _v12_domain_features(
        test_headers,
        test_ids,
    )

    train_series = _rad_pd.read_csv(
        ROOT / "train_series.csv",
        dtype={
            "StudyInstanceUID": str,
            "SeriesInstanceUID": str,
        },
    )
    test_series = _rad_pd.read_csv(
        ROOT / "test_series.csv",
        dtype={
            "StudyInstanceUID": str,
            "SeriesInstanceUID": str,
        },
    )
    test_frame = _rad_pd.read_csv(
        ROOT / "test.csv",
        dtype={
            "StudyInstanceUID": str
        },
    )

    train_protocol = (
        _v10_protocol_features(
            train_series,
            data["ids"],
        )
    )
    test_protocol = (
        _v10_protocol_features(
            test_series,
            test_ids,
        )
    )
    train_demographics = (
        _v10_demographic_features(
            data["train"],
            data["ids"],
        )
    )
    test_demographics = (
        _v10_demographic_features(
            test_frame,
            test_ids,
        )
    )

    weak_conf = (
        0.30
        + 1.70
        * _rad_np.abs(
            y - 0.5
        )
        * 2.0
    )
    weak_conf = _rad_np.clip(
        weak_conf,
        0.30,
        2.0,
    )

    unique_folds = sorted(
        int(value)
        for value in _rad_np.unique(
            fold[gold]
        )
        if int(value) >= 0
    )

    if len(unique_folds) < 3:
        return (
            None,
            _rad_np.zeros(
                len(_RAD_LABELS),
                _rad_np.float64,
            ),
            ["none"] * len(_RAD_LABELS),
        )

    ridge_cross = _rad_np.full(
        y.shape,
        _rad_np.nan,
        _rad_np.float64,
    )
    hgb_cross = _rad_np.full(
        y.shape,
        _rad_np.nan,
        _rad_np.float64,
    )
    pair_cross = _rad_np.full(
        y.shape,
        _rad_np.nan,
        _rad_np.float64,
    )

    for outer in unique_folds:
        train_mask = (
            fold != int(outer)
        )
        valid_mask = (
            gold
            & (fold == int(outer))
        )

        if not valid_mask.any():
            continue

        graph = _v12_stable_label_graph(
            y,
            data["base"],
            fold,
            train_mask,
        )

        x_fold = _v12_design(
            data["base"],
            data["public"],
            data["pass2"],
            train_protocol,
            train_demographics,
            train_domain,
            graph,
        )

        for target in range(
            len(_RAD_LABELS)
        ):
            target_y = y[
                :, target
            ]
            weight = weak_conf[
                :, target
            ].copy()
            weight[gold] = 20.0

            ridge = _v10_fit_ridge(
                x_fold[train_mask],
                target_y[train_mask],
                weight[train_mask],
                alpha=26.0,
            )
            ridge_cross[
                valid_mask,
                target,
            ] = _v10_predict_ridge(
                ridge,
                x_fold[valid_mask],
            )

            try:
                hgb = _v12_fit_domain_hgb(
                    x_fold[train_mask],
                    target_y[train_mask],
                    weight[train_mask],
                    seed=1201
                    + 29 * int(outer)
                    + target,
                )
                hgb_cross[
                    valid_mask,
                    target,
                ] = hgb.predict(
                    x_fold[
                        valid_mask
                    ]
                )
            except Exception:
                hgb_cross[
                    valid_mask,
                    target,
                ] = ridge_cross[
                    valid_mask,
                    target,
                ]

            try:
                pair = _v12_fit_pairwise(
                    x_fold,
                    target_y,
                    train_mask,
                    gold,
                    seed=6011
                    + 43 * int(outer)
                    + target,
                )
                pair_prediction = (
                    _v12_predict_pairwise(
                        pair,
                        x_fold[
                            valid_mask
                        ],
                    )
                )
                if pair_prediction is None:
                    raise RuntimeError(
                        "pairwise unavailable"
                    )

                pair_cross[
                    valid_mask,
                    target,
                ] = pair_prediction
            except Exception:
                pair_cross[
                    valid_mask,
                    target,
                ] = ridge_cross[
                    valid_mask,
                    target,
                ]

    matrices = [
        ridge_cross[
            gold_index
        ],
        hgb_cross[
            gold_index
        ],
        pair_cross[
            gold_index
        ],
    ]

    if not all(
        _rad_np.isfinite(
            matrix
        ).all()
        for matrix in matrices
    ):
        return (
            None,
            _rad_np.zeros(
                len(_RAD_LABELS),
                _rad_np.float64,
            ),
            ["none"] * len(_RAD_LABELS),
        )

    ridge_gold = _v10_rank(
        matrices[0]
    )
    hgb_gold = _v10_rank(
        matrices[1]
    )
    pair_gold = _v10_rank(
        matrices[2]
    )

    # Two diverse low-capacity mixtures.
    rank_blend_gold = _v10_rank(
        0.55 * ridge_gold
        + 0.45 * pair_gold
    )
    domain_blend_gold = _v10_rank(
        0.46 * ridge_gold
        + 0.24 * hgb_gold
        + 0.30 * pair_gold
    )

    candidate_gold = {
        "ridge_domain": ridge_gold,
        "hgb_domain": hgb_gold,
        "pair_auc": pair_gold,
        "rank_blend": rank_blend_gold,
        "domain_blend": domain_blend_gold,
    }

    deployment = _rad_np.zeros(
        len(_RAD_LABELS),
        _rad_np.float64,
    )
    chosen_model = [
        "none"
        for _ in _RAD_LABELS
    ]

    grid = _rad_np.asarray(
        [
            0.05,
            0.08,
            0.12,
            0.16,
            0.20,
            0.24,
            0.28,
            0.32,
        ],
        _rad_np.float64,
    )

    for target in range(
        len(_RAD_LABELS)
    ):
        truth = gold_y[
            :, target
        ]
        base = v11_gold[
            :, target
        ]

        base_auc = _v10_target_auc(
            truth,
            base,
        )
        if not _rad_np.isfinite(
            base_auc
        ):
            continue

        positive_count = int(
            (truth > 0.5).sum()
        )
        negative_count = int(
            (truth <= 0.5).sum()
        )

        best = None

        for model_name, matrix in (
            candidate_gold.items()
        ):
            for mix in grid:
                candidate = (
                    (1.0 - mix)
                    * base
                    + mix
                    * matrix[
                        :, target
                    ]
                )

                auc = _v10_target_auc(
                    truth,
                    candidate,
                )
                if not _rad_np.isfinite(
                    auc
                ):
                    continue

                support = (
                    _v10_bootstrap_positive_fraction(
                        truth,
                        base,
                        candidate,
                        seed=3109
                        + 101 * target
                        + int(
                            1000 * mix
                        ),
                        repeats=320,
                    )
                )

                (
                    valid_folds,
                    fold_positive,
                    fold_median,
                ) = _v11_fold_consistency(
                    truth,
                    base,
                    candidate,
                    gold_fold,
                )

                complexity = {
                    "ridge_domain": 0.0,
                    "pair_auc": 0.0001,
                    "rank_blend": 0.0002,
                    "hgb_domain": 0.0007,
                    "domain_blend": 0.0005,
                }[model_name]

                objective = (
                    auc
                    - 0.0018
                    * float(mix)
                    - complexity
                )

                record = {
                    "model": model_name,
                    "mix": float(mix),
                    "auc": float(auc),
                    "gain": float(
                        auc - base_auc
                    ),
                    "support": float(
                        support
                    ),
                    "valid_folds": int(
                        valid_folds
                    ),
                    "fold_positive": float(
                        fold_positive
                    ),
                    "fold_median": float(
                        fold_median
                    ),
                    "positive_count":
                        positive_count,
                    "negative_count":
                        negative_count,
                    "objective": float(
                        objective
                    ),
                }

                if (
                    best is None
                    or record[
                        "objective"
                    ]
                    > best[
                        "objective"
                    ]
                ):
                    best = record

        if best is None:
            continue

        class_support = (
            min(
                best[
                    "positive_count"
                ],
                best[
                    "negative_count"
                ],
            )
        )

        fold_ok = (
            best["valid_folds"] < 2
            or best[
                "fold_positive"
            ] >= 0.60
        )

        strong = (
            best["gain"] >= 0.0045
            and best["support"] >= 0.63
            and fold_ok
            and class_support >= 5
        )
        moderate = (
            best["gain"] >= 0.0020
            and best["support"] >= 0.595
            and fold_ok
            and class_support >= 4
        )

        if strong:
            deployment[target] = min(
                best["mix"],
                0.28,
            )
            chosen_model[target] = (
                best["model"]
            )
        elif moderate:
            deployment[target] = min(
                best["mix"],
                0.10,
            )
            chosen_model[target] = (
                best["model"]
            )

    if not (
        deployment > 0
    ).any():
        return (
            None,
            deployment,
            chosen_model,
        )

    full_mask = _rad_np.ones(
        len(y),
        dtype=bool,
    )
    full_graph = _v12_stable_label_graph(
        y,
        data["base"],
        fold,
        full_mask,
    )

    x_train = _v12_design(
        data["base"],
        data["public"],
        data["pass2"],
        train_protocol,
        train_demographics,
        train_domain,
        full_graph,
    )
    x_test = _v12_design(
        test_base,
        test_public,
        test_pass2,
        test_protocol,
        test_demographics,
        test_domain,
        full_graph,
    )

    challenger = _rad_np.zeros(
        (
            len(test_ids),
            len(_RAD_LABELS),
        ),
        _rad_np.float64,
    )

    for target in range(
        len(_RAD_LABELS)
    ):
        model_name = chosen_model[
            target
        ]

        if model_name == "none":
            challenger[
                :, target
            ] = test_base[
                :, target
            ]
            continue

        target_y = y[
            :, target
        ]
        weight = weak_conf[
            :, target
        ].copy()
        weight[gold] = 20.0

        ridge = _v10_fit_ridge(
            x_train,
            target_y,
            weight,
            alpha=26.0,
        )
        ridge_test = _v10_predict_ridge(
            ridge,
            x_test,
        )

        pair = None
        pair_test = ridge_test

        if model_name in (
            "pair_auc",
            "rank_blend",
            "domain_blend",
        ):
            try:
                pair = _v12_fit_pairwise(
                    x_train,
                    target_y,
                    _rad_np.ones(
                        len(y),
                        dtype=bool,
                    ),
                    gold,
                    seed=8311 + target,
                )
                predicted = (
                    _v12_predict_pairwise(
                        pair,
                        x_test,
                    )
                )
                if predicted is not None:
                    pair_test = predicted
            except Exception:
                pair_test = ridge_test

        hgb_test = ridge_test

        if model_name in (
            "hgb_domain",
            "domain_blend",
        ):
            try:
                hgb = _v12_fit_domain_hgb(
                    x_train,
                    target_y,
                    weight,
                    seed=9109 + target,
                )
                hgb_test = hgb.predict(
                    x_test
                )
            except Exception:
                hgb_test = ridge_test

        if model_name == "ridge_domain":
            output = ridge_test
        elif model_name == "hgb_domain":
            output = hgb_test
        elif model_name == "pair_auc":
            output = pair_test
        elif model_name == "rank_blend":
            output = (
                0.55 * ridge_test
                + 0.45 * pair_test
            )
        else:
            output = (
                0.46 * ridge_test
                + 0.24 * hgb_test
                + 0.30 * pair_test
            )

        challenger[
            :, target
        ] = output

    return (
        _v10_rank(
            challenger
        ),
        deployment,
        chosen_model,
    )



def _rad_main_v12():
    work = _RadPath("/kaggle/working")
    primary = work / "submission.csv"

    test = _rad_pd.read_csv(
        ROOT / "test.csv",
        dtype={"StudyInstanceUID": str},
    )
    expected_ids = test.StudyInstanceUID.astype(str).tolist()

    baseline = _rad_pd.read_csv(
        primary,
        dtype={"StudyInstanceUID": str},
    )
    _rad_validate(baseline, expected_ids)

    device = _rad_torch.device("cuda:0")

    test_series = _rad_pd.read_csv(
        ROOT / "test_series.csv",
        dtype={
            "StudyInstanceUID": str,
            "SeriesInstanceUID": str,
        },
    )
    plane = dict(
        zip(
            test_series.SeriesInstanceUID,
            test_series.Anatomical_Plane,
        )
    )

    # One deterministic header pass is shared by all Rad views and V12.
    test_header_table = _v12_header_table(
        "test_series"
    )

    def cache(slots, crop, tag, threshold):
        globals().update(
            SLOTS=list(slots),
            N_SLOT=len(slots),
            CACHE_SLICES=8,
            IMG=224,
            CACHE_IMG=224,
            CROP_MM=float(crop),
            RULES=dict(RULES_LEGACY),
        )

        headers = test_header_table.copy()
        studies, pixels, masks = build_cache(
            pick_slots(headers, plane),
            plane,
            lat_of(headers, tag + " "),
            tag,
        )

        positions = {
            str(uid): index
            for index, uid in enumerate(studies)
        }
        missing = [
            uid
            for uid in expected_ids
            if uid not in positions
        ]
        if missing:
            raise RuntimeError(
                f"{len(missing)} studies absent from {tag}"
            )

        order = _rad_np.asarray(
            [positions[uid] for uid in expected_ids],
            dtype=_rad_np.int64,
        )
        pixels = pixels[order]
        masks = masks[order]

        tokens = int(
            _rad_np.repeat(
                masks[:, :, None],
                CACHE_SLICES,
                axis=2,
            ).sum()
        )

        if tokens < int(
            threshold
            * len(test)
            * N_SLOT
            * CACHE_SLICES
        ):
            raise RuntimeError(
                f"insufficient slices for {tag}: {tokens}"
            )

        return pixels, masks

    encoder_path = _rad_find_file(
        "ResNet50.pt",
        _RAD_ENCODER_SHA256,
    )
    encoder = _RadEncoder()
    encoder.load_state_dict(
        _rad_torch.load(
            encoder_path,
            map_location="cpu",
            weights_only=True,
        ),
        strict=True,
    )
    encoder.eval().to(device)

    for parameter in encoder.parameters():
        parameter.requires_grad_(False)

    if _rad_torch.cuda.device_count() > 1:
        encoder = _rad_nn.DataParallel(
            encoder,
            device_ids=list(range(_rad_torch.cuda.device_count())),
        )

    public_slots = [
        ("SAG_FS", "Sagittal", None, True),
        ("COR_FS", "Coronal", None, True),
        ("AX_FS", "Axial", None, True),
    ]

    pixels, masks = cache(
        public_slots,
        10000.0,
        "test-v12-public",
        0.85,
    )

    public_heads, public_path = _rad_load_public_heads(
        device,
        _RAD_REFERENCE_HEADS_SHA256,
    )

    features, token_mask = _rad_encode(
        encoder,
        pixels,
        masks,
        device,
    )

    public_predictions = [
        _rad_predict_head(
            head,
            features,
            token_mask,
            device,
        )
        for head in public_heads
    ]

    (
        public_probability_rank,
        public_consensus,
        public_stability,
    ) = _v8_consensus(public_predictions)

    del public_heads, features, token_mask, pixels, masks
    _rad_gc.collect()
    _rad_torch.cuda.empty_cache()

    globals().update(
        SLOTS=list(_RAD_E13_SLOTS),
        N_SLOT=len(_RAD_E13_SLOTS),
        CACHE_SLICES=_RAD_E13_CACHE_SLICES,
        IMG=_RAD_E13_IMG,
        CACHE_IMG=_RAD_E13_IMG,
        CROP_MM=_RAD_E13_CROP_MM,
        RULES=dict(RULES_LEGACY),
    )

    e13_heads, e13_path = _rad_load_e13_heads(device)

    pixels, masks = cache(
        _RAD_E13_SLOTS,
        _RAD_E13_CROP_MM,
        "test-v12-e13",
        0.85,
    )

    features, token_mask = _rad_encode(
        encoder,
        pixels,
        masks,
        device,
    )

    e13_predictions = [
        _rad_predict_head(
            head,
            features,
            token_mask,
            device,
        )
        for head in e13_heads
    ]

    (
        e13_probability_rank,
        e13_consensus,
        e13_stability,
    ) = _v8_consensus(e13_predictions)

    del features, token_mask, pixels, masks
    _rad_gc.collect()
    _rad_torch.cuda.empty_cache()

    public_metric = _v8_bundle_metrics(_RadPath(public_path))
    e13_metric = _v8_bundle_metrics(_RadPath(e13_path))

    public_oof = _v8_discover_oof_auc("public")
    e13_oof = _v8_discover_oof_auc("e13")

    if public_oof is not None and e13_oof is not None:
        diff = _rad_np.nan_to_num(
            e13_oof - public_oof,
            nan=0.0,
        )
        e13_weight = _rad_np.clip(
            0.50 + 0.55 * diff,
            0.42,
            0.58,
        )
    elif public_metric is not None and e13_metric is not None:
        diff = _rad_np.nan_to_num(
            e13_metric - public_metric,
            nan=0.0,
        )
        e13_weight = _rad_np.clip(
            0.50 + 0.45 * diff,
            0.43,
            0.57,
        )
    else:
        relative = (
            _v8_robust_z(e13_stability, floor=0.02)
            - _v8_robust_z(public_stability, floor=0.02)
        )
        e13_weight = _rad_np.clip(
            0.50 + 0.025 * _rad_np.tanh(relative),
            0.465,
            0.535,
        )

    reference_adaptive = _rad_rank_columns(
        (1.0 - e13_weight)[None, :] * public_consensus
        + e13_weight[None, :] * e13_consensus
    )

    reference_exact = _rad_rank_columns(
        (1.0 - _RAD_E13_MEMBER_WEIGHT) * public_probability_rank
        + _RAD_E13_MEMBER_WEIGHT * e13_probability_rank
    )

    baseline_rank = _rad_rank_columns(
        baseline[_RAD_LABELS].to_numpy()
    )
    parent_specialist = _v8_parent_specialist(baseline_rank)

    exact_e10 = baseline_rank.copy()
    for index, target in enumerate(_RAD_LABELS):
        if target not in _RAD_EXCLUDE:
            exact_e10[:, index] = (
                (1.0 - _RAD_ALPHA) * baseline_rank[:, index]
                + _RAD_ALPHA * reference_exact[:, index]
            )
    exact_e10_rank = _rad_rank_columns(exact_e10)

    adaptive_e10 = parent_specialist.copy()

    ref_quality = _rad_np.maximum(
        public_stability,
        e13_stability,
    )
    ref_z = _v8_robust_z(ref_quality, floor=0.02)
    e10_alpha = _rad_np.clip(
        0.50 + 0.025 * _rad_np.tanh(ref_z),
        0.47,
        0.53,
    )

    for index, target in enumerate(_RAD_LABELS):
        if target not in _RAD_EXCLUDE:
            adaptive_e10[:, index] = (
                (1.0 - e10_alpha[index]) * parent_specialist[:, index]
                + e10_alpha[index] * reference_adaptive[:, index]
            )

    adaptive_e10_rank = _rad_rank_columns(adaptive_e10)

    pixels, masks = cache(
        _RAD_E11_SLOTS,
        _RAD_E11_CROP_MM,
        "test-v12-pass2",
        0.55,
    )

    features, token_mask = _rad_encode(
        encoder,
        pixels,
        masks,
        device,
    )

    pass2_predictions = [
        _rad_predict_head(
            head,
            features,
            token_mask,
            device,
        )
        for head in e13_heads
    ]

    (
        pass2_probability_rank,
        pass2_consensus,
        pass2_stability,
    ) = _v8_consensus(pass2_predictions)

    pass2_z = _v8_robust_z(pass2_stability, floor=0.02)

    diversity = _rad_np.zeros(
        len(_RAD_LABELS),
        _rad_np.float64,
    )

    for j in range(len(_RAD_LABELS)):
        x = pass2_consensus[:, j]
        y = adaptive_e10_rank[:, j]
        corr = (
            float(_rad_np.corrcoef(x, y)[0, 1])
            if _rad_np.std(x) > 0 and _rad_np.std(y) > 0
            else 1.0
        )
        diversity[j] = 1.0 - abs(corr)

    diversity_z = _v8_robust_z(diversity, floor=0.01)

    pass2_alpha = _rad_np.clip(
        _RAD_V48_SECOND_ALPHA
        + 0.022 * _rad_np.tanh(pass2_z)
        + 0.010 * _rad_np.tanh(diversity_z),
        0.105,
        0.205,
    )

    priors = {
        "ACL": 0.020,
        "MCL": -0.010,
        "Medial Meniscus": 0.005,
        "Lateral Meniscus": 0.005,
        "Medial OA": 0.020,
        "Lateral OA": 0.020,
        "PF OA": 0.010,
        "Effusion": -0.020,
        "Synovitis": -0.015,
        "Baker's": 0.000,
        "Contusion": -0.010,
        "Fracture": 0.020,
    }

    for j, target in enumerate(_RAD_LABELS):
        pass2_alpha[j] = float(
            _rad_np.clip(
                pass2_alpha[j] + priors.get(target, 0.0),
                0.095,
                0.215,
            )
        )

    exact_final = _rad_rank_columns(
        (1.0 - _RAD_V48_SECOND_ALPHA) * exact_e10_rank
        + _RAD_V48_SECOND_ALPHA * pass2_probability_rank
    )

    adaptive_final = _rad_rank_columns(
        (1.0 - pass2_alpha)[None, :] * adaptive_e10_rank
        + pass2_alpha[None, :] * pass2_consensus
    )

    # V8 is the measured high-performing anchor.
    v8_anchor = _rad_rank_columns(
        0.65 * exact_final
        + 0.35 * adaptive_final
    )

    # New trained ensemble layer. It uses only genuine OOF train
    # predictions for fitting and cross-fitted official labels for
    # deciding whether each target is allowed to change at all.
    meta_rank, meta_weight = _v10_meta_stack(
        test_base=baseline_rank,
        test_public=public_probability_rank,
        test_pass2=pass2_probability_rank,
        test_ids=expected_ids,
    )

    if meta_rank is None:
        v10_anchor = v8_anchor
    else:
        v10_anchor = _rad_rank_columns(
            (
                1.0
                - meta_weight
            )[None, :]
            * v8_anchor
            + meta_weight[None, :]
            * meta_rank
        )

    # V11 is a challenger above the measured 0.921 V10 route.
    # If its evidence is not strong enough, final_rank remains V10 exactly.
    (
        v11_rank,
        v11_weight,
        v11_model,
    ) = _v11_meta_challenger(
        test_base=baseline_rank,
        test_public=public_probability_rank,
        test_pass2=pass2_probability_rank,
        test_ids=expected_ids,
    )

    if v11_rank is None:
        v11_anchor = v10_anchor
    else:
        v11_anchor = _rad_rank_columns(
            (
                1.0
                - v11_weight
            )[None, :]
            * v10_anchor
            + v11_weight[None, :]
            * v11_rank
        )

    (
        v12_rank,
        v12_weight,
        v12_model,
    ) = _v12_meta_challenger(
        test_base=baseline_rank,
        test_public=public_probability_rank,
        test_pass2=pass2_probability_rank,
        test_ids=expected_ids,
    )

    if v12_rank is None:
        final_rank = v11_anchor
    else:
        final_rank = _rad_rank_columns(
            (
                1.0
                - v12_weight
            )[None, :]
            * v11_anchor
            + v12_weight[None, :]
            * v12_rank
        )

    final = baseline.copy()
    final[_RAD_LABELS] = final_rank

    _rad_validate(final, expected_ids)
    final.to_csv(primary, index=False)

    for candidate in work.glob("submission*.csv"):
        if candidate.name != "submission.csv":
            try:
                candidate.unlink()
            except OSError:
                pass

    for candidate in work.glob("*diagnostics*.csv"):
        try:
            candidate.unlink()
        except OSError:
            pass

    del (
        encoder,
        e13_heads,
        features,
        token_mask,
        pixels,
        masks,
        public_predictions,
        e13_predictions,
        pass2_predictions,
    )
    _rad_gc.collect()
    _rad_torch.cuda.empty_cache()


_rad_main_v12()
